# IberoamericaBooks — original development notebook (sanitized)

> **Historical / Read-only Reference**

This file is a near-verbatim, sanitized copy of the original development notebook. It preserves the original cell order, code, detailed diagnoses and reasoning so the evolution of the ETL can be inspected without exposing private project data.

Sanitization is intentionally limited to:

- removing saved outputs, execution counters and transient notebook metadata;
- replacing private row examples, internal identifiers and private report names with explicit placeholders;
- adding this explanatory header.

The private input workbooks used during development are not distributed, so this historical notebook is **not intended to be executed as-is**. For a reproducible workflow with public fixtures and live OpenAlex enrichment, use the [portfolio demo notebook](../iberoamerica_books_demo.ipynb).


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import re
import unicodedata
import openpyxl
import tabulate
import json
import requests
from difflib import SequenceMatcher
from pathlib import Path

from io import StringIO
from xlsx2csv import Xlsx2csv


### 1) Creacion de la base de datos

In [ ]:
# Creamos la base de datos que iremos rellenando
con = sqlite3.connect("iberoamerica_libros.sqlite")
cur = con.cursor()

In [ ]:
# Para testear, borramos las tablas de la base de datos (eliminar este codigo al terminar de testear)
# cur.executescript("""
#                   DROP TABLE IF EXISTS isbn;                  
#                   DROP TABLE IF EXISTS isbn_autores;                  
#                   DROP TABLE IF EXISTS libros;                  
#                   DROP TABLE IF EXISTS isbn_libros;                   
#                   DROP TABLE IF EXISTS libros_autores;                  
#                   DROP TABLE IF EXISTS ventas;
#                   DROP TABLE IF EXISTS bibliotecas;
#                   DROP TABLE IF EXISTS editoriales;                  
#                   DROP TABLE IF EXISTS isbn_editoriales;
#                   DROP TABLE IF EXISTS autores;
#                   """)

In [ ]:
# Creamos las tablas de la base de datos
cur.executescript("""

PRAGMA foreign_keys = ON;

-- ================
-- TABLAS MAESTRAS
-- ================

CREATE TABLE IF NOT EXISTS editoriales (
  id_editorial INTEGER PRIMARY KEY AUTOINCREMENT,
  nombre TEXT UNIQUE NOT NULL,
  pais TEXT,
  tipo TEXT,
  cantidad_isbn_doi INTEGER
);

CREATE TABLE IF NOT EXISTS autores (
  id_autor INTEGER PRIMARY KEY AUTOINCREMENT,
  nombre TEXT UNIQUE NOT NULL,
  afiliacion TEXT,
  pais TEXT,
  genero TEXT
);

CREATE TABLE IF NOT EXISTS libros (
  id_libro INTEGER PRIMARY KEY AUTOINCREMENT,
  titulo TEXT,
  colaboradores TEXT
);
                  
-- Entidad central
CREATE TABLE IF NOT EXISTS isbn (
  id_isbn INTEGER PRIMARY KEY AUTOINCREMENT,
  excel_origen TEXT,
  id_original TEXT,
  isbn13 TEXT UNIQUE,
  formato_publicacion TEXT,
  titulo TEXT,
  anio_publicacion INTEGER,
  colaboradores TEXT,
  pais_publicacion TEXT,
  ciudad_publicacion TEXT,
  editorial_principal_nombre TEXT,
  coedicion_si_no TEXT,
  institucion_coeditora_nombre TEXT, 
  tipo_coedicion TEXT,
  resumen TEXT,
  palabras_clave TEXT,
  idioma TEXT,
  coleccion TEXT,
  bisac TEXT,
  thema TEXT,
  dewey TEXT,
  tipo_colaboracion TEXT,
  tipo_obra TEXT,
  doi TEXT,
  doi_simeh TEXT,
  doi_scielo TEXT,
  doi_openalex TEXT,
  openalex_consulted TEXT,
  citacion INTEGER,          
  news_mentions INTEGER,
  blog_mentions INTEGER,
  policy_mentions INTEGER,
  patents_mentions INTEGER,
  x_mentions INTEGER,
  peer_reviews_mentions INTEGER,
  weibo_mentions INTEGER,
  facebook_mentions INTEGER,
  wikipedia_mentions INTEGER,
  google_plus_mentions INTEGER,
  linkedin_mentions INTEGER,
  reddit_mentions INTEGER,
  pinterest_mentions INTEGER,
  f1000_mentions INTEGER,
  q_a_mentions INTEGER,
  video_mentions INTEGER,
  clinical_guidelines_mentions INTEGER,
  bluesky_mentions INTEGER,
  podcast_mentions INTEGER,
  syllabi_mentions INTEGER,
  mendeley_readers INTEGER,
  dimensions_citations INTEGER                  
);

-- ==========
-- DEPENDIENTES
-- ==========

-- 1:N ventas por isbn
CREATE TABLE IF NOT EXISTS ventas (
  id_isbn INTEGER NOT NULL,
  anio_venta INTEGER,
  canal TEXT,
  tienda TEXT,
  modalidad TEXT,
  pais_venta TEXT,
  area TEXT,
  subarea TEXT,
  formato_comercial TEXT,
  edicion TEXT,
  cantidad_vendida INTEGER,
  FOREIGN KEY (id_isbn) REFERENCES isbn(id_isbn) ON DELETE CASCADE
);

-- 1:N bibliotecas por isbn
CREATE TABLE IF NOT EXISTS bibliotecas (
  id_biblio INTEGER PRIMARY KEY AUTOINCREMENT,
  nombre TEXT,
  pais TEXT,
  id_isbn INTEGER NOT NULL,
  anio_incorporacion INTEGER,
  FOREIGN KEY (id_isbn) REFERENCES isbn(id_isbn) ON DELETE CASCADE
);

-- N:M isbn–autores (con rol)
CREATE TABLE IF NOT EXISTS isbn_autores (
  id_isbn  INTEGER NOT NULL,
  id_autor  INTEGER NOT NULL,
  rol_autor TEXT NOT NULL DEFAULT '',
  PRIMARY KEY (id_isbn, id_autor, rol_autor),
  FOREIGN KEY (id_isbn) REFERENCES isbn(id_isbn) ON DELETE CASCADE,
  FOREIGN KEY (id_autor) REFERENCES autores(id_autor) ON DELETE CASCADE
);
                
-- N:M isbn–editoriales (coedición, distribución, etc.)
CREATE TABLE IF NOT EXISTS isbn_editoriales (
  id_isbn      INTEGER NOT NULL,
  id_editorial  INTEGER NOT NULL,
  rol_editorial TEXT NOT NULL DEFAULT '',
  PRIMARY KEY (id_isbn, id_editorial, rol_editorial),
  FOREIGN KEY (id_isbn) REFERENCES isbn(id_isbn) ON DELETE CASCADE,
  FOREIGN KEY (id_editorial) REFERENCES editoriales(id_editorial) ON DELETE CASCADE
);
                  
-- N:M isbn–libros
CREATE TABLE IF NOT EXISTS isbn_libros (
  id_isbn      INTEGER NOT NULL,
  id_libro  INTEGER NOT NULL,
  PRIMARY KEY (id_isbn, id_libro),
  FOREIGN KEY (id_isbn) REFERENCES isbn(id_isbn) ON DELETE CASCADE,
  FOREIGN KEY (id_libro) REFERENCES libros(id_libro) ON DELETE CASCADE
);
                  
-- N:M libros–autores
CREATE TABLE IF NOT EXISTS libros_autores (
  id_libro  INTEGER NOT NULL,
  id_autor  INTEGER NOT NULL,
  PRIMARY KEY (id_libro, id_autor),
  FOREIGN KEY (id_libro) REFERENCES libros(id_libro) ON DELETE CASCADE,
  FOREIGN KEY (id_autor) REFERENCES autores(id_autor) ON DELETE CASCADE
);


-- Eliminar AUTORES que queden sin referencias en NINGUNA tabla puente
CREATE TRIGGER IF NOT EXISTS trg_cleanup_autores_after_isbn_autores_delete
AFTER DELETE ON isbn_autores
FOR EACH ROW
BEGIN
  DELETE FROM autores
  WHERE id_autor = OLD.id_autor
    AND NOT EXISTS (SELECT 1 FROM isbn_autores  WHERE id_autor = OLD.id_autor)
    AND NOT EXISTS (SELECT 1 FROM libros_autores WHERE id_autor = OLD.id_autor);       
END;
                  

-- Eliminar EDITORIALES que queden sin referencias (solo se relacionan vía isbn_editoriales)
CREATE TRIGGER IF NOT EXISTS trg_cleanup_editoriales_after_isbn_editoriales_delete
AFTER DELETE ON isbn_editoriales
FOR EACH ROW
BEGIN
  DELETE FROM editoriales
  WHERE id_editorial = OLD.id_editorial
    AND NOT EXISTS (SELECT 1 FROM isbn_editoriales WHERE id_editorial = OLD.id_editorial);
END;

-- Eliminar LIBROS que queden sin referencias en isbn_libros
-- (las filas en libros_autores se borran por ON DELETE CASCADE)
CREATE TRIGGER IF NOT EXISTS trg_cleanup_libros_after_isbn_libros_delete
AFTER DELETE ON isbn_libros
FOR EACH ROW
BEGIN
  DELETE FROM libros
  WHERE id_libro = OLD.id_libro
    AND NOT EXISTS (SELECT 1 FROM isbn_libros WHERE id_libro = OLD.id_libro);
END;

""")

### 2) Dataframes

En esta parte del script vamos a limpiar las tablas procedentes de distintas instituciones para cargar la información en la base de datos.

#### Dataframe (simeh)
Esta información procede de **SIMEH** (https://simeh.co/). SIMEH será nuestra principal fuente de datos para alimentar la base de datos: ofrece una lista de publicaciones (por ISBN) por editorial.

Lo primero es limpiar/preparar los datos de la tabla de SIMEH para que estén en el formato adecuado para ser introducidos en la base de datos.

##### Información a limpiar:

- **Eliminamos las filas** que tienen `NA` en todas sus columnas
- **Tratar las celdas que están vacías** (incluido espacio en blanco) para que cuenten como **NA**

- **Columna "Colaborador"**
    - Sustituir `|` por `;`.
    - Eliminar espacios al principio y final del *string*.
    - Eliminar espacios antes y después de `;`.

- **Columna "Fecha de publicación"**
    - Las filas que tienen formato `dd/mm/yyyy` pasarlas a `yyyy`.
    - Nos quedamos con las obras publicadas entre 2020 y 2024

- **Columna "THEMA"**
    - Cuando un libro tiene mas de una categoría en THEMA, el valor se reparte en varias filas (1 categoría por fila). Solucionar esto: poner las categorías en una sola fila, separadas con `;`.

- **Columna "Texto de contenido"**
    - Hay resúmenes que están partidos en varias filas. Arreglar.

- **Columna "Editores"**
    - Eliminar el contenido entre paréntesis y los propios paréntesis.
    - Sustituir `|` por `;`.
    - Eliminar espacios al principio y final del *string*.
    - Eliminar espacios antes y después de `;`.
    - Buscar la editorial que se encuentra en la columna **Editor** en esta columna **Editores**.  Si se encuentra: eliminar la editorial de la columna **Editores**.

- **Columnas "BISAC, THEMA, Dewey, Palabras claves"**:
    - Sustituir `|` por `;`.
    - Eliminar espacios al principio y final del *string*.
    - Eliminar espacios antes y después de `;`.

- **Las filas que tienen ISBN13** duplicados las unimos en una sola fila, de manera que las celdas que tienen distintos valores quedarían separadas con `;` en la misma celda
    - Dentro de cada celda, eliminamos los strings repetidos

- **Columna "ISBN13"**
    - Las celdas que estan vacías (es decir, que falta la información relativa al ISBN13), rellenarlas con `<falta_isbn13>`
    - Hay celdas que NO SON ISBN13 pero están metidos como tal. Estos casos los vamos a eliminar ya que darán ruido en fases posteriores. Se va a generar un archivo excel aparte que almacene estos casos para comunicarle a las editoriales estas erratas.

- **Columna "Título"**:
    - Las celdas que estan vacías (es decir, que falta el título del libro), rellenarlas con `<falta_titulo_libro>`


##### Limitaciones de la tabla:

- **Colaborador** : en esta columna nos encontramos con valores que limitan la automatización:
    - a veces aparecen valores como "varios autores"
    - autores que a veces tienen asociado texto entre paréntesis. Esto da lugar a que un mismo autor sea considerado como dos autores distintos.
        - ejemplo: "[NOMBRE COMPLETO] (Compilador) / [NOMBRE COMPLETO]"
    - autores que a veces aparecen con un apellido y otras con los dos. Da lugar al mismo problema mencionado arriba.
        - ejemplos: "[NOMBRE ABREVIADO] / [NOMBRE COMPLETO]"; "[NOMBRE CON UN APELLIDO] / [NOMBRE CON DOS APELLIDOS]"; "[NOMBRE ABREVIADO] / [NOMBRE COMPLETO]"
    - autores que a veces aparecen con nombramientos como "Ph. D", "Msc", "Arq" y otras veces no. Da lugar al mismo problema mencionado arriba.
        - ejemplos:  "[NOMBRE] Ph. D / [NOMBRE]"; "Arq. [NOMBRE] / [NOMBRE]"

- **Editores**: a veces ocurre que el nombre de la editorial principal (de la columna **Editor**) aparece repetido en esta columna **Editores**

In [ ]:
carpeta = Path("input_data/simeh")

simeh_files = list()

for item in carpeta.glob("*.xlsx"):
    if item.name.startswith("~$"):       # evita temporales de Excel
        continue
    print(item.name)
    simeh_files.append(item)

print()
print(f"{len(simeh_files)} excels a procesar...")
print()
print()
print()

In [ ]:
print("########## COMENZANDO LIMPIEZA ##########")
count = 0

simeh_files_done = list()

for item in carpeta.glob("*.xlsx"):
    if item.name.startswith("~$"):       # evita temporales de Excel
        continue

    count = count + 1
    print(f"{count}) Limpiando el excel: '{item}'")

    buf = StringIO()
    Xlsx2csv(item, outputencoding="utf-8").convert(buf)  # primera hoja
    buf.seek(0)

    simeh = pd.read_csv(buf, dtype=str)  # <- ya como DataFrame
    # Eliminamos las filas que tienen NA en todas sus columnas
    simeh = simeh.dropna(how="all")








    # Limpiando la columna "Fecha de publicación"
    # Las filas que tienen formato “dd/mm/yyyy” pasarlas a “yyyy”.

    # 1) Normalizamos a string
    simeh["Fecha de publicación"] = simeh["Fecha de publicación"].astype("string").str.strip()

    # 2) Intentamos parsear fechas (día/mes/año) y sacar el año. Guardamos como numerico pero con formato integer anulable de pandas ("Int64"), ya que esta columna tiene valores NA (tipo pd.NA) y "astype(int)" no admite NAtype (te daria error si usas "astype(int)"). Esta alternativa "astype("Int64")" permite una variante de entero que admite faltantes (pd.NA). No es lo mismo que el int64 de NumPy.
    y1 = pd.to_datetime(simeh["Fecha de publicación"], dayfirst=True, errors="coerce").dt.year.astype("Int64")

    # 3) Para lo que no se pudo parsear, extraemos un año 19xx/20xx del texto
    y2 = simeh["Fecha de publicación"].str.extract(r'((?:19|20)\d{2})', expand=False).astype("Int64")

    # 4) Combinamos manteniendo el "Int64"
    simeh["Fecha de publicación"] = y1.fillna(y2)








    # Nos quedamos solo con las filas que tienen en la columna "Fecha de publicación" valores entre 2020 y 2024 (estos años incluidos)
    simeh = simeh[simeh["Fecha de publicación"].between(2020, 2024)]
    # Limpiando la columna "Colaborador"
    # Sustituir "|" por ";"
    simeh["Colaborador"] = simeh["Colaborador"].str.replace("|", ";")
    # Eliminar espacios al principio y final del string
    simeh["Colaborador"] = simeh["Colaborador"].str.strip()
    # Eliminar espacios antes y despues de ";"
    simeh["Colaborador"] = simeh["Colaborador"].str.replace(r"\s*;\s*", ";", regex=True)










    # Limpiando la columna "THEMA"

    # 1) Rellenar RecordReference hacia abajo en filas con texto pero sin ID
    mask_cont = simeh["RecordReference"].isna() & simeh["THEMA"].notna()
    simeh.loc[mask_cont, "RecordReference"] = simeh["RecordReference"].ffill()

    # 2) Utilidades
    def _split_thema(val): # Convierte THEMA en lista de códigos, separando por ';' y limpiando espacios.
        if pd.isna(val):
            return []
        s = str(val).strip()
        if not s:
            return []
        return [x.strip() for x in s.split(';') if x.strip()]

    def _row_full(r: pd.Series) -> bool: # Devuelve True si TODAS las columnas de la fila están rellenas: - No NaN; - Si es string, no vacío ni espacios
        for v in r:
            if pd.isna(v):
                return False
            if isinstance(v, str) and (v.strip() == ""):
                return False
        return True

    # 3) Precalcular qué filas están "completas"
    mask_fila_completa = simeh.apply(_row_full, axis=1)

    # 4) Consolidar THEMA dentro de cada RecordReference
    # Conservaremos la primera fila COMPLETA dentro del grupo (si existe);
    # si no existe ninguna completa, usaremos la primera fila del grupo.
    indices_a_conservar = []
    indices_a_eliminar = []

    for rr, g in simeh.groupby("RecordReference", sort=False):
        # THEMA únicos (orden de aparición preservado)
        tema_list = []
        for v in g["THEMA"]:
            tema_list.extend(_split_thema(v))
        tema_unicos = list(dict.fromkeys(tema_list))  # preserva orden

        # Selección de la fila destino
        idx_completas = g.index[mask_fila_completa.reindex(g.index, fill_value=False)]
        if len(idx_completas) > 0:
            idx_destino = idx_completas[0]  # primera completa del grupo
        else:
            idx_destino = g.index[0]        # si ninguna completa, la primera del grupo

        # Escribir THEMA consolidado en la fila destino
        simeh.at[idx_destino, "THEMA"] = ";".join(tema_unicos) if tema_unicos else np.nan

        # Guardar índices a conservar/eliminar si deseas desduplicar
        indices_a_conservar.append(idx_destino)
        indices_a_eliminar.extend([i for i in g.index if i != idx_destino])

    # 5) Elimina filas cuyo RecordReference esté repetido y THEMA tenga algo (no NaN ni vacío)
    def _has_thema(val): # True si THEMA tiene contenido útil (no NaN, no vacío, no solo ';' o espacios)
        if pd.isna(val):
            return False
        s = str(val).replace(';', '').strip()
        return len(s) > 0

    # Duplicado de RecordReference (conserva la primera aparición)
    mask_dup_rr = simeh.duplicated("RecordReference", keep="first")

    # THEMA con contenido
    mask_thema_val = simeh["THEMA"].apply(_has_thema)

    # Filas a eliminar: duplicadas por RR y con THEMA con contenido
    to_drop = simeh.index[mask_dup_rr & mask_thema_val]

    # Eliminar
    simeh = simeh.drop(index=to_drop)












    # Limpiando la columna "Texto de contenido"

    # 1) Rellenar RecordReference hacia abajo en filas con texto pero sin ID
    mask_cont = simeh["RecordReference"].isna() & simeh["Texto de contenido"].notna()
    simeh.loc[mask_cont, "RecordReference"] = simeh["RecordReference"].ffill()

    # 2) Utilidades
    def _clean_text(s): # Limpia espacios y unifica saltos; devuelve '' si NaN.
        if pd.isna(s):
            return ""
        s = str(s).strip()
        # Colapsar múltiples espacios y saltos de línea en un solo espacio
        s = " ".join(s.split())
        return s

    def _row_full(r: pd.Series) -> bool: # True si TODAS las columnas están rellenas: No NaN; Si es string, no vacío tras strip
        for v in r:
            if pd.isna(v):
                return False
            if isinstance(v, str) and (v.strip() == ""):
                return False
        return True

    # 3) Precalcular filas completas
    mask_fila_completa = simeh.apply(_row_full, axis=1)

    # 4) Consolidar 'Texto de contenido' dentro de cada RecordReference
    indices_destino = []     # para saber qué fila conservar como principal
    indices_contenido_otros = []  # para (opcional) eliminarlas después

    for rr, g in simeh.groupby("RecordReference", sort=False):
        # Limpia y recoge piezas de texto en orden de aparición (omitimos vacíos)
        piezas = [t for t in g["Texto de contenido"].map(_clean_text) if t]
        if piezas:
            # Evitar duplicados exactos preservando orden
            piezas_unicas = list(dict.fromkeys(piezas))
            texto_consolidado = " ".join(piezas_unicas)
        else:
            texto_consolidado = np.nan

        # Elegir fila destino: primera COMPLETA; si no hay, la primera del grupo
        idx_completas = g.index[mask_fila_completa.reindex(g.index, fill_value=False)]
        idx_destino = idx_completas[0] if len(idx_completas) > 0 else g.index[0]
        indices_destino.append(idx_destino)

        # Escribir el texto consolidado en la fila destino
        simeh.at[idx_destino, "Texto de contenido"] = texto_consolidado

        # Guardar el resto de filas con contenido para (opcional) eliminarlas
        otros = g.index.difference([idx_destino])
        # De esos otros, nos quedamos con los que tengan contenido no vacío
        otros_con_contenido = [
            i for i in otros
            if _clean_text(simeh.at[i, "Texto de contenido"]) != ""
        ]
        indices_contenido_otros.extend(otros_con_contenido)

    # 5) Quedarte con UNA fila por RecordReference (la consolidada)
    #     Elimina las filas "fragmento" que tenían contenido y no son la destino
    simeh = simeh.drop(index=indices_contenido_otros).reset_index(drop=True)











    # Limpiando la columna "Editores"
    # Eliminar el contenido entre paréntesis y los propios paréntesis
    simeh["Editores"] = simeh["Editores"].str.replace(r"\([^)]*\)", "", regex=True).str.strip()

    # Sustituir "|" por ";"
    simeh["Editores"] = simeh["Editores"].str.replace("|", ";")

    # Eliminar espacios al principio y final del string
    simeh["Editores"] = simeh["Editores"].str.strip()

    # Eliminar espacios antes y despues de ";"
    simeh["Editores"] = simeh["Editores"].str.replace(r"\s*;\s*", ";", regex=True)










    # Las filas que tienen ISBN13 duplicados las unimos en una sola fila, de manera que las celdas que tienen distintos valores quedarían separadas con `;` en la misma celda

    # 1) Normalizar ISBN13 en una línea:
    # - lo pasamos a string
    # - quitamos espacios
    # - quitamos un ".0" final típico de Excel cuando guardó como float (antes vimos que si, que la columna "ISBN13" es tipo float)
    simeh["ISBN13"] = (
        simeh["ISBN13"]
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    # 2) Agregador sencillo: une valores únicos en orden con ';' (si todos son iguales, quedará solo uno)
    def join_uniques(col: pd.Series) -> pd.Series:
        vals = col.dropna().astype(str).str.strip()
        if vals.empty:
            return pd.NA
        return ";".join(pd.unique(vals))

    # 3) Partimos en con/sin ISBN13 para no “agrupar” los NaN
    con_isbn = simeh[simeh["ISBN13"].notna()].copy()
    sin_isbn = simeh[simeh["ISBN13"].isna()].copy()

    # 4) Agregamos por ISBN13 aplicando la misma regla a todas las columnas
    agrupado = (
        con_isbn
        .groupby("ISBN13", dropna=False)
        .agg(join_uniques)
        .reset_index()
    )

    # 5) Unimos de nuevo las filas sin ISBN13 (se quedan tal cual)
    simeh = pd.concat([agrupado, sin_isbn], ignore_index=True)










    # Limpiando las columnas "BISAC, THEMA, Dewey, Palabras claves":
    # Eliminar el contenido entre paréntesis y los propios paréntesis
    simeh["BISAC"] = simeh["BISAC"].str.replace(r"\([^)]*\)", "", regex=True).str.strip()
    simeh["THEMA"] = simeh["THEMA"].str.replace(r"\([^)]*\)", "", regex=True).str.strip()
    simeh["Dewey"] = simeh["Dewey"].str.replace(r"\([^)]*\)", "", regex=True).str.strip()
    simeh["Palabras claves"] = simeh["Palabras claves"].str.replace(r"\([^)]*\)", "", regex=True).str.strip()

    # Sustituir "|" por ";"
    simeh["BISAC"] = simeh["BISAC"].str.replace("|", ";")
    simeh["THEMA"] = simeh["THEMA"].str.replace("|", ";")
    simeh["Dewey"] = simeh["Dewey"].str.replace("|", ";")
    simeh["Palabras claves"] = simeh["Palabras claves"].str.replace("|", ";")

    # Eliminar espacios al principio y final del string
    simeh["BISAC"] = simeh["BISAC"].str.strip()
    simeh["THEMA"] = simeh["THEMA"].str.strip()
    simeh["Dewey"] = simeh["Dewey"].str.strip()
    simeh["Palabras claves"] = simeh["Palabras claves"].str.strip()

    # Eliminar espacios antes y despues de ";"
    simeh["BISAC"] = simeh["BISAC"].str.replace(r"\s*;\s*", ";", regex=True)
    simeh["THEMA"] = simeh["THEMA"].str.replace(r"\s*;\s*", ";", regex=True)
    simeh["Dewey"] = simeh["Dewey"].str.replace(r"\s*;\s*", ";", regex=True)
    simeh["Palabras claves"] = simeh["Palabras claves"].str.replace(r"\s*;\s*", ";", regex=True)










    # Quitar duplicados dentro de cada celda: si en una celda tenemos valores repetidos separados por ; y queremos quedarnos con strings únicos
    cols = ["Formato", "Editor", "Colaborador","BISAC", "THEMA", "Dewey", "Palabras claves", "Editores", "Fecha de publicación"]

    def dedup_semicolon_cell(x):
        if pd.isna(x) or x == "":
            return pd.NA
        items = [t.strip() for t in str(x).split(";") if t.strip() != ""]
        seen = []
        for t in items:
            if t not in seen:
                seen.append(t)
        return ";".join(seen) if seen else pd.NA

    for c in cols:
        simeh[c] = simeh[c].apply(dedup_semicolon_cell)









        
    # Quitar de la columna "Editores" el valor que está en "Editor" (si existe allí).
    def _norm(s: str) -> str:
        # normaliza: quita espacios duplicados y compara en minúsculas
        return re.sub(r"\s+", " ", s).strip().casefold()

    def limpiar_editores_row(editor, editores):
        if pd.isna(editores):
            return editores  # no hay nada que limpiar

        # lista de editores en "Editores"
        L_editores = [e.strip() for e in str(editores).split(";") if e.strip()]

        # conjunto de nombres presentes en "Editor" (puede haber varios)
        S_editor = set()
        if pd.notna(editor):
            S_editor = {_norm(e) for e in str(editor).split(";") if e.strip()}

        # filtra los de "Editores" que NO estén en "Editor" (comparación normalizada)
        L_filtrada = [e for e in L_editores if _norm(e) not in S_editor]

        return ";".join(L_filtrada) if L_filtrada else np.nan  # deja NaN si queda vacío

    simeh["Editores"] = simeh.apply(
        lambda r: limpiar_editores_row(r["Editor"], r["Editores"]),
        axis=1
    )










    # Tratar las celdas que están vacías (incluido espacio en blanco) para que cuenten como NA

    # Convertir valores faltantes de pandas (NaN/NaT) a None de Python
    simeh = simeh.where(simeh.notna(), None)
    # Reemplazar strings vacíos o solo espacios por None
    simeh = simeh.replace(r'^\s*$', None, regex=True)
    # Rellenar todo lo que sea "None/NA" en la columna "ISBN13" con "<falta_isbn13>"
    simeh["ISBN13"] = simeh["ISBN13"].fillna("<falta_isbn13>")
    # Rellenar todo lo que sea "None/NA" en la columna "Titulo" con "<falta_titulo_libro>"
    simeh["Título"] = simeh["Título"].fillna("<falta_titulo_libro>")


    simeh_files_done.append(item)

    print("########## LIMPIEZA TERMINADA ##########")
    print(f"Excel '{item}' limpiado. Cargando los datos limpios a la base de datos...")

    simeh.to_excel("clean_simeh.xlsx", sheet_name="sheet1", index=False)

    print()
    print("########## CARGANDO A LA BASE DE DATOS ##########")



    for index, row in simeh.iterrows():
        nombre_editoriales = row.loc["Editor"]

        institucion_coeditora_nombre = row.loc["Editores"]
        
        colaboradores = row.loc["Colaborador"]

        id_original = row.loc["RecordReference"]
        titulo = row.loc["Título"]
        subtitulo = row.loc["Subtítulo"]

        if subtitulo is not None:
            titulo = titulo + ": " + subtitulo
        
        try: pais_publicacion = row.loc["País de publicación"]
        except: pais_publicacion = None

        try: ciudad_publicacion = row.loc["Ciudad de publicación"]
        except: ciudad_publicacion = None

        anio_publicacion = row.loc["Fecha de publicación"]
        isbn13 = row.loc["ISBN13"]
        idioma = row.loc["Idioma"]
        coleccion = row.loc["Colección"]

        bisac = row.loc["BISAC"]
        ##print("bisac_orig: ", bisac)
        if bisac is not None:
            bisac_list = list()
            if re.findall(";", bisac):
                bisac_split = bisac.split(";")
                for bisac_item in bisac_split:
                    bisac = re.findall("^\[?([A-Z]{3}\d{6})\]?", bisac_item)[0] # Nos quedamos solo con el codigo alfanumerico
                    bisac_list.append(bisac)
                bisac = ";".join(bisac_list)
            else:
                bisac = re.findall("^\[?([A-Z]{3}\d{6})\]?", bisac)[0] # Nos quedamos solo con el codigo alfanumerico
        ##print("bisac_mod: ", bisac)
        ##print()

        thema = row.loc["THEMA"]
        ## print("thema_orig: ", thema)
        if thema is not None:
            thema_list = list()
            if re.findall(";", thema):
                thema_split = thema.split(";")
                for thema_item in thema_split:
                    thema = re.findall("\[([A-Z0-9;]+)\]", thema_item)[0] # Nos quedamos solo con el codigo alfanumerico
                    thema_list.append(thema)
                thema = ";".join(thema_list)
            else:
                thema = re.findall("\[([A-Z0-9;]+)\]", thema)[0] # Nos quedamos solo con el codigo alfanumerico
        ## print("thema_mod: ", thema)
        ## print()

        dewey = row.loc["Dewey"]
        ## print("dewey_orig: ", dewey)

        if dewey is None:
            dewey = None
        else:
            dewey_list = []
            text = str(dewey)

            if ";" in text:
                dewey_split = text.split(";")
                for dewey_item in dewey_split:
                    matches = re.findall(r"\d{3}(?:\.\d+)?", str(dewey_item))
                    if matches:
                        dewey_list.append(matches[0])
                    else:
                        # print("  [AVISO] Fragmento sin código Dewey reconocible:", repr(dewey_item))
                        pass
                dewey = ";".join(dewey_list) if dewey_list else None
            else:
                matches = re.findall(r"\d{3}(?:\.\d+)?", text)
                dewey = matches[0] if matches else None

        ## print("dewey_mod: ", dewey)
        ## print()




        palabras_clave = row.loc["Palabras claves"]
        resumen = row.loc["Texto de contenido"]
        
        if institucion_coeditora_nombre is not None:
            coedicion_si_no = "si"
        else:
            coedicion_si_no = "no"
        
        formato_publicacion = row.loc["Formato"]
        doi_simeh = row.loc["DOI"]

        rol_autor = "autor"

        ##### Rellenando la tabla "isbn" de la base de datos #####
        cur.execute("INSERT OR IGNORE INTO isbn (excel_origen, id_original, titulo, pais_publicacion, ciudad_publicacion, anio_publicacion, colaboradores, isbn13, idioma, coleccion, bisac, thema, dewey, editorial_principal_nombre, palabras_clave, resumen, coedicion_si_no, institucion_coeditora_nombre, formato_publicacion, doi_simeh) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", (item.name, id_original, titulo, pais_publicacion, ciudad_publicacion, anio_publicacion, colaboradores, isbn13, idioma, coleccion, bisac, thema, dewey, nombre_editoriales, palabras_clave, resumen, coedicion_si_no, institucion_coeditora_nombre, formato_publicacion, doi_simeh, ))
        con.commit()

        cur.execute("SELECT id_isbn FROM isbn WHERE isbn13 = ?", (isbn13, ))
        id_isbn = cur.fetchone()[0]
        # print(id_isbn)

        ##### Rellenando la tabla "editoriales" de la base de datos #####
        # Aqui hay que tener en cuenta las variables "nombre_editoriales" y "institucion_coeditora_nombre", que pueden presentar varias opciones separadas con ";"
        
        nombre_editoriales = nombre_editoriales.split(";") # convertimos a lista

        if institucion_coeditora_nombre is None:
            editoriales_total = nombre_editoriales
        else:
            institucion_coeditora_nombre = institucion_coeditora_nombre.split(";")
            editoriales_total = nombre_editoriales + institucion_coeditora_nombre

        for editorial in editoriales_total:
            cur.execute("INSERT OR IGNORE INTO editoriales (nombre) VALUES (?)", (editorial, ))
            con.commit()

            cur.execute("SELECT id_editorial FROM editoriales WHERE nombre = ?", (editorial, ))
            id_editorial = cur.fetchone()[0]
            # print(id_editorial)

        ##### Rellenando la tabla "isbn_editoriales" de la base de datos #####
            if editorial in nombre_editoriales:
                rol_editorial = "editor"
                cur.execute("INSERT OR IGNORE INTO isbn_editoriales (id_isbn, id_editorial, rol_editorial) VALUES (?,?,?)", (id_isbn, id_editorial, rol_editorial, ))
            else:
                rol_editorial = "coeditor"
                cur.execute("INSERT OR IGNORE INTO isbn_editoriales (id_isbn, id_editorial, rol_editorial) VALUES (?,?,?)", (id_isbn, id_editorial, rol_editorial, ))
            con.commit()


        ##### Rellenando la tabla "autores" de la base de datos #####
        # Aqui hay que tener en cuenta las variables "colaboradores" que pueden presentar varios colaboradores separados con ";"

        colaboradores_split = colaboradores.split(";")

        for colaborador in colaboradores_split:
            cur.execute("INSERT OR IGNORE INTO autores (nombre) VALUES (?)", (colaborador, ))
            con.commit()

            cur.execute("SELECT id_autor FROM autores WHERE nombre = ?", (colaborador, ))
            id_autor = cur.fetchone()[0]

        ##### Rellenando la tabla "isbn_autores" de la base de datos #####
            cur.execute("INSERT OR IGNORE INTO isbn_autores (id_isbn, id_autor, rol_autor) VALUES (?,?,?)", (id_isbn, id_autor, rol_autor, ))
            con.commit()
        

        ##### Rellenando la tabla "libros" de la base de datos #####
        # Aqui hay que tener en cuenta que en el dataframe "simeh", un titulo de libro puede estar repetido en varias filas (porque un libro puede tener distintos isbn, y en este df simeh tenemos que 1 fila = 1 isbn), pero tambien hay que considerar que pueden existir dos libros distintos con el mismo titulo. Para tratar de minimizar estos casos, vamos a hacer un query que sea "titulo + colaboradores", y si esta combinacion ya se encuentra en la base de datos, el libro no se introduce
        cur.execute("SELECT titulo, colaboradores FROM libros WHERE titulo = ? AND colaboradores = ?", (titulo, colaboradores, ))
        coincidencia_sql = cur.fetchone()
        # print(coincidencia_sql)
        if coincidencia_sql is None:
            cur.execute("INSERT INTO libros (titulo, colaboradores) VALUES (?,?)", (titulo, colaboradores, ))
            con.commit()

        ##### Rellenando la tabla "isbn_libros" de la base de datos #####
        cur.execute("SELECT id_libro FROM libros WHERE titulo = ? AND colaboradores = ?", (titulo, colaboradores, ))
        id_libro = cur.fetchone()[0]
        cur.execute("INSERT OR IGNORE INTO isbn_libros (id_isbn, id_libro) VALUES (?,?)", (id_isbn, id_libro))
        con.commit()


        ##### Rellenando la tabla "libros_autores" de la base de datos #####   
        # De nuevo, como en el dataframe "simeh" tenemos que un libro puede estar repetido varias veces, usamos el truco de antes: query que sea "titulo + colaboradores", y si esta combinacion ya se encuentra en la base de datos, el libro no se introduce. (No hace falta hacer otra vez el query: esta guardado en el objeto "coincidencia_sql").
        # Tambien hay que tener en cuenta las variables "colaboradores" que pueden presentar varios colaboradores separados con ";"
        if coincidencia_sql is None:
            for colaborador in colaboradores_split:
                cur.execute("SELECT id_autor FROM autores WHERE nombre = ?", (colaborador, ))
                id_autor = cur.fetchone()[0]

                cur.execute("INSERT OR IGNORE INTO libros_autores (id_libro, id_autor) VALUES (?,?)", (id_libro, id_autor, ))
                con.commit()


    print()
    print("########## CARGA TERMINADA ##########")
    print(f"Excel '{item}' cargado en la base de datos. Pasando al siguiente excel...")
    print()
    print()
    print()
    print()



print()
print(f"Todos los archivos ({len(simeh_files_done)}/{len(simeh_files)}) de la carpeta han sido procesados: ")
for item in simeh_files_done:
    print("          ", item.name)

#### Dataframe (plantilla_propia)
En este dataframe, tenemos que 1 fila = 1 titulo de libro

##### Información a limpiar:
- **Columna "anio_publicacion"**:  
    - Filtramos por los años 2020, 2021, 2022, 2024
- **Columnas "ISBN IMPRESO, ISBN PDF, ISBN EPUB"**:
    - Si un titulo tiene mas de 1 ISBN, duplicamos la fila.
        - Para la fila correspondiente con el "ISBN IMPRESO", en la columna "formato_publicacion" pondremos "impreso"
        - Para la fila correspondiente con el "ISBN PDF", en la columna "formato_publicacion" pondremos "pdf"
        - Para la fila correspondiente con el "ISBN EPUB", en la columna "formato_publicacion" pondremos "ebook"
    - Sin embargo, no duplicaremos el DOI

- **Columnas "Autor, Editor, Compilador, Coordinador"**:
    - A veces los nombres estan en formato "Nombre Apellido", pero la mayoría de las veces esta en formato "Apellidos, Nombre"
    - Eliminar los espacios antes y despues del ";"

- **Columna "palabras_clave"**:
    - Eliminar los espacios antes y despues del ";"


##### Limitaciones de la tabla:





In [ ]:
carpeta = Path("input_data/plantilla_propia")

propia_files = list()

for item in carpeta.glob("*.xlsx"):
    if item.name.startswith("~$"):       # evita temporales de Excel
        continue
    print(item.name)
    propia_files.append(item)

print()
print(f"{len(propia_files)} excels a procesar...")
print()
print()
print()

In [ ]:
print("########## COMENZANDO LIMPIEZA ##########")
print()
count = 0

propia_files_done = list()

for item in carpeta.glob("*.xlsx"):
    if item.name.startswith("~$"):       # evita temporales de Excel
        continue

    count = count + 1
    file_name = item
    print(f"{count}) Limpiando el excel: '{file_name}'")

    propia = pd.read_excel(file_name, sheet_name=None)
    propia = propia["Libros"]
    # Nos quedamos solo con las filas que tienen en la columna "anio_publicacion" valores entre 2020 y 2024 (estos años incluidos)
    propia = propia[propia["anio_publicacion"].between(2020, 2024)]
    # Eliminamos los espacios antes y despues del ";" en las columnas "Autor", "Editor", "Compilador", "Coordinador"
    cols = ["Autor", "Editor", "Compilador", "Coordinador"]

    for col in cols:
        if col in propia.columns:
            propia[col] = (
                propia[col]
                .astype("string")                          # asegura tipo texto
                .str.replace(r"\s*;\s*", ";", regex=True)  # quita espacios antes y después de ;
                .str.strip()                               # quita espacios al inicio y final de la celda
            )
    # Eliminamos los espacios antes y despues del ";" en la columna "palabras_clave"
    for col in ["palabras_clave"]:
        if col in propia.columns:
            propia[col] = (
                propia[col]
                .astype("string")                          # asegura tipo texto
                .str.replace(r"\s*;\s*", ";", regex=True)  # quita espacios antes y después de ;
                .str.strip()                               # quita espacios al inicio y final de la celda
            )






    # Duplicamos las filas en funcion de los valores de las columnas "ISBN IMPRESO, ISBN PDF, ISBN EPUB", pero prestando atencion a dos detalles importantes:
    #   1) Las filas duplicadas solo deben tener un valor de ISBN asociado. Es decir, si tenemos, por ejemplo, que la primera fila tiene rellenada esas tres columnas, duplicamos la fila tres veces, pero en la fila original deben borrarse los valores de las columnas "ISBN PDF" e "ISBN EPUD"; la segunda fila repetida deben borrarse los valores de las columnas "ISBN IMPRESO" e "ISBN EPUB"; y la tercera fila repetida los valores de las columnas "ISBN IMPRESO" e "ISBN PDF".
    #   2) De las filas duplicadas, solo la original debe conservar el doi. El doi deben eliminarse de las duplicadas

    # Nombre de la columna de DOI en tu Excel
    doi_col = "DOI"   # cámbialo a "doi" o como se llame en tu dataframe

    # 0. Guardamos el índice original para saber qué filas vienen de la misma fila de origen
    propia = propia.reset_index().rename(columns={"index": "orig_row"})

    # Columnas de ISBN
    isbn_cols = ["ISBN IMPRESO", "ISBN PDF", "ISBN EPUB"]

    # Resto de columnas que queremos replicar tal cual (incluye orig_row y DOI)
    id_cols = [c for c in propia.columns if c not in isbn_cols]

    # 1. Pasamos de formato ancho a largo: una fila por cada ISBN no vacío
    propia_long = (
        propia
        .melt(
            id_vars=id_cols,
            value_vars=isbn_cols,
            var_name="tipo_isbn",
            value_name="isbn_valor"
        )
    )

    # Nos quedamos solo con los ISBN que tienen contenido (no NaN y no cadena vacía)
    propia_long = propia_long[
        propia_long["isbn_valor"].notna() &
        (propia_long["isbn_valor"].astype(str).str.strip() != "")
    ]

    # 2. Creamos las columnas de salida con solo un ISBN por fila
    for col in isbn_cols:
        propia_long[col] = None

    # Asignamos el valor de ISBN solo a la columna correspondiente
    for col in isbn_cols:
        mask = propia_long["tipo_isbn"] == col
        propia_long.loc[mask, col] = propia_long.loc[mask, "isbn_valor"]

    # 3. Borrar el DOI de las filas duplicadas (solo la "original" lo mantiene)
    # Ordenamos para que la "original" (la primera) quede bien definida
    propia_long = propia_long.sort_values(["orig_row", "tipo_isbn"])

    # Para cada fila original (orig_row), marcamos las duplicadas
    mask_dup = propia_long.duplicated(subset=["orig_row"], keep="first")

    # En las duplicadas, borramos el DOI
    propia_long.loc[mask_dup, doi_col] = None

    # 4. Construimos el dataframe final
    # Quitamos la columna auxiliar 'tipo_isbn' y 'isbn_valor'
    cols_final = [c for c in id_cols if c != "orig_row"] + isbn_cols
    propia = propia_long[cols_final].copy()






    

    # Pasamos las columnas "ISBN IMPRESO, ISBN PDF, ISBN EPUB" al tipo "Int64" para eliminar el float final ".0"
    propia["ISBN IMPRESO"] = propia["ISBN IMPRESO"].astype("Int64")
    propia["ISBN PDF"] = propia["ISBN PDF"].astype("Int64")
    propia["ISBN EPUB"] = propia["ISBN EPUB"].astype("Int64")


    



    # # Tratar las celdas que están vacías (incluido espacio en blanco) o que tienen NaN (el NA de pandas) a None de Python:
    # 1) Unificar las celdas vacías como valores faltantes
    propia = propia.replace(r'^\s*$', pd.NA, regex=True)
    # 2) Pasar todos los NA (NaN, NaT, pd.NA) a None de Python
    #   Importante: convertir antes a 'object' si quieres que se conserven los None
    propia = propia.astype("object").where(propia.notna(), None)


    propia_files_done.append(item)

    print("########## LIMPIEZA TERMINADA ##########")
    print(f"Excel '{item}' limpiado. Cargando los datos limpios a la base de datos...")

    
    propia.to_excel("clean_propia.xlsx", index=False)



    print()
    print("########## CARGANDO A LA BASE DE DATOS ##########")

    for index, row in propia.iterrows():
        if row.loc["ISBN IMPRESO"] is not None:
            isbn13 = row.loc["ISBN IMPRESO"]
            formato_publicacion = "impreso"
        elif row.loc["ISBN PDF"] is not None:
            isbn13 = row.loc["ISBN PDF"]
            formato_publicacion = "pdf"
        elif row.loc["ISBN EPUB"] is not None:
            isbn13 = row.loc["ISBN EPUB"]
            formato_publicacion = "ebook"
            
        titulo = row.loc["Titulo"]
        anio_publicacion = row.loc["anio_publicacion"]

        # Cuidado: en este dataframe, hay autores con dos formatos distintos: "Nombre Apellidos" y "Apellidos, Nombre". Tenemos que lidiar con esto y ponerlo todo en el mismo formato: "Nombre Apellidos"
        autores = row.loc["Autor"]

        ##print(titulo, isbn13, formato_publicacion)
        ##print()
        #print("autor_original: ", autores)
        autores_clean = list()

        if autores is not None:
            #print(autores)
            if re.findall(";", autores):
                autores_split = autores.split(";")

                for autor in autores_split:
                    if re.findall(",", autor):
                        autor_split = autor.split(",")
                        #print(autor_split)
                        autor = autor_split[1].strip() + " " + autor_split[0].strip()
                        #print(autor)
                    autores_clean.append({
                        "rol":"autor",
                        "name": autor
                    })
                ##print(autores_clean)
                ##print()

            else:
                if re.findall(",", autores):
                    autores_split = autores.split(",")
                    #print(autores_split)
                    autores = autores_split[1].strip() + " " + autores_split[0].strip()
                    
                autores_clean.append({
                    "rol":"autor",
                    "name":autores.strip()
                })
        else:
            pass


        # Ahora toca hacer lo mismo con las columnas "Editor, Compilador, Coordinador", y al final unificar todos los diccionarios en uno
        editores = row.loc["Editor"]
        
        ##print(editores)

        editores_clean = list()

        if editores is not None:
            if re.findall(";", editores):
                editores_split = editores.split(";")

                for editor in editores_split:
                    if re.findall(",", editor):
                        editor_split = editor.split(",")
                        editor = editor_split[1].strip() + " " + editor_split[0].strip()
                    editores_clean.append({
                        "rol":"editor",
                        "name": editor
                    })
                ##print(editores_clean)
                ##print()
            else:
                if re.findall(",", editores):
                    editores_split = editores.split(",")
                    #print(editores_split)
                    editores = editores_split[1].strip() + " " + editores_split[0].strip()

                editores_clean.append({
                    "rol":"editor",
                    "name":editores.strip()
                })
        else:
            pass


        
        compiladores = row.loc["Compilador"]
        ##print(compiladores)

        compiladores_clean = list()

        if compiladores is not None:
            if re.findall(";", compiladores):
                compiladores_split = compiladores.split(";")

                for compilador in compiladores_split:
                    if re.findall(",", compilador):
                        compilador_split = compilador.split(",")
                        compilador = compilador_split[1].strip() + " " + compilador_split[0].strip()
                    compiladores_clean.append({
                        "rol":"compilador",
                        "name": compilador
                    })
                ##print(compiladores_clean)
                ##print()
            else:
                if re.findall(",", compiladores):
                    compiladores_split = compiladores.split(",")
                    #print(compiladores_split)
                    compiladores = compiladores_split[1].strip() + " " + compiladores_split[0].strip()

                compiladores_clean.append({
                    "rol":"compilador",
                    "name":compiladores.strip()
                })
        else:
            pass





        coordinadores = row.loc["Coordinador"]
        ## print(coordinadores)

        coordinadores_clean = list()

        if coordinadores is not None:
            if re.findall(";", coordinadores):
                coordinadores_split = coordinadores.split(";")

                for coordinador in coordinadores_split:
                    if re.findall(",", coordinador):
                        coordinador_split = coordinador.split(",")
                        coordinador = coordinador_split[1].strip() + " " + coordinador_split[0].strip()
                    coordinadores_clean.append({
                        "rol":"coordinador",
                        "name": coordinador
                    })
                ##print(coordinadores_clean)
                ##print()
            else:
                if re.findall(",", coordinadores):
                    coordinadores_split = coordinadores.split(",")
                    #print(coordinadores_split)
                    coordinadores = coordinadores_split[1].strip() + " " + coordinadores_split[0].strip()

                coordinadores_clean.append({
                    "rol":"coordinador",
                    "name":coordinadores.strip()
                })
        else:
            pass



        # Juntamos los diccionarios de los autores, editores, compiladores y coordinadores en un solo diccionario llamado "colaboradores_dict"
        
        colaboradores_roles_list = autores_clean + editores_clean + compiladores_clean + coordinadores_clean

        colaboradores = list()

        for colaborador in colaboradores_roles_list:
            name = colaborador["name"]
            colaboradores.append(name)

        colaboradores = ";".join(colaboradores)

        if len(colaboradores) < 1:
            continue
        




        pais_publicacion = row.loc["pais_publicacion"]
        ciudad_publicacion = row.loc["ciudad_publicacion"]
        editorial_principal_nombre = row.loc["editorial_principal_nombre"]

        coedicion_si_no = row.loc["coedicion_si_no"]
        if coedicion_si_no is not None: 
            coedicion_si_no = coedicion_si_no.lower()
        # print(coedicion_si_no)

        if coedicion_si_no is None or re.findall("no", coedicion_si_no):
            coedicion_si_no = "no"
            institucion_coeditora_nombre = None
            tipo_coedicion = None

        else:
            coedicion_si_no = "si"
            institucion_coeditora_nombre = row.loc["institucion_coeditora_nombre"]
            tipo_coedicion = row.loc["tipo_coedicion"]

        # print(coedicion_si_no, institucion_coeditora_nombre, tipo_coedicion)

        tipo_obra = row.loc["tipo_obra"]
        resumen = row.loc["resumen"]
        palabras_clave = row.loc["palabras_clave"]
        idioma = row.loc["idioma"]
        coleccion = row.loc["coleccion"]
        bisac = row.loc["bisac"]
        if bisac is not None:
            bisac = re.findall("^([A-Z]{3}\d{6})\s*>", bisac)[0]
        ##print(bisac)






        thema_raw = row.loc["thema"]
        print("thema_orig:", thema_raw)

        thema = None  # valor por defecto

        if thema_raw is not None and not pd.isna(thema_raw):
            text = str(thema_raw).strip()
            codes = []

            # 1) Caso con corchetes: [P] ..., [P;PBA] ...
            m = re.search(r"\[([A-Z0-9;]+)\]", text)
            if m:
                codes_raw = m.group(1)  # "P" o "P;PBA"
                for c in codes_raw.split(";"):
                    c = c.strip()
                    if c and re.fullmatch(r"[A-Z0-9]{1,6}", c):
                        codes.append(c)

            # 2) Caso tipo: QD > Filosofía; QDTN; QDT > Temas de la filosofía; DS > ...
            parts = [p.strip() for p in text.split(";") if p.strip()]
            for part in parts:
                before_gt = part.split(">")[0].strip()  # "QD", "QDTN", "QDT", "DS", etc.
                if re.fullmatch(r"[A-Z0-9]{1,6}", before_gt) and before_gt not in codes:
                    codes.append(before_gt)

            if codes:
                thema = ";".join(codes)

        print("thema_mod:", thema)
        print()
                
        



        
        dewey_raw = row.loc["dewey"]
        print("dewey_orig: ", dewey_raw)

        dewey = None  # valor por defecto

        if dewey_raw is not None:
            text = str(dewey_raw)

            # Captura un código Dewey al principio de la cadena, con o sin corchetes:
            # [370.7] ...  → 370.7
            # 100 Filosofía y psicología; ... → 100
            m = re.search(r"^\s*\[?(\d{3}(?:\.\d+)?)\]?", text)
            if m:
                dewey = m.group(1)

        print("dewey_mod: ", dewey)
        print()



        doi = row.loc["DOI"]
        if doi is not None:
            try:
                doi = re.findall(".org/(.*)", doi)[0]
                #print("doi: ", doi)
            except:
                doi = None
        
        

        ##### Rellenando la tabla "isbn" de la base de datos #####
        cur.execute("INSERT OR IGNORE INTO isbn (excel_origen, isbn13, formato_publicacion, titulo, anio_publicacion, colaboradores, pais_publicacion, ciudad_publicacion, editorial_principal_nombre, coedicion_si_no, institucion_coeditora_nombre, tipo_coedicion, resumen, palabras_clave, idioma, coleccion, bisac, thema, dewey, tipo_obra, doi) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", (item.name, isbn13, formato_publicacion, titulo, anio_publicacion, colaboradores, pais_publicacion, ciudad_publicacion, editorial_principal_nombre, coedicion_si_no, institucion_coeditora_nombre, tipo_coedicion, resumen, palabras_clave, idioma, coleccion, bisac, thema, dewey, tipo_obra, doi, ))
        con.commit()

        cur.execute("SELECT id_isbn FROM isbn WHERE isbn13 = ?", (isbn13, ))
        id_isbn = cur.fetchone()[0]
        # print(id_isbn)



        ##### Rellenando la tabla "editoriales" de la base de datos #####
        if institucion_coeditora_nombre is None:
            editoriales_total = [editorial_principal_nombre]
        else:
            editoriales_total = [editorial_principal_nombre, institucion_coeditora_nombre]
            
        ##print(editoriales_total)

        for editorial in editoriales_total:
            cur.execute("INSERT OR IGNORE INTO editoriales (nombre) VALUES (?)", (editorial, ))
            con.commit()

            cur.execute("SELECT id_editorial FROM editoriales WHERE nombre = ?", (editorial, ))
            id_editorial = cur.fetchone()[0]
            # print(id_editorial)

        
        ##### Rellenando la tabla "isbn_editoriales" de la base de datos #####
            if editorial in editorial_principal_nombre:
                rol_editorial = "editor"
                cur.execute("INSERT OR IGNORE INTO isbn_editoriales (id_isbn, id_editorial, rol_editorial) VALUES (?,?,?)", (id_isbn, id_editorial, rol_editorial, ))
            else:
                rol_editorial = "coeditor"
                cur.execute("INSERT OR IGNORE INTO isbn_editoriales (id_isbn, id_editorial, rol_editorial) VALUES (?,?,?)", (id_isbn, id_editorial, rol_editorial, ))
            con.commit()


        ##### Rellenando la tabla "autores" de la base de datos #####
        # Aqui hay que tener en cuenta las variables "colaboradores" que pueden presentar varios colaboradores separados con ";"

        ##print(colaboradores_roles_list)

        for colaborador in colaboradores_roles_list:
            rol_autor = colaborador["rol"]
            colaborador = colaborador["name"]
            #print(rol_autor, colaborador)

            cur.execute("INSERT OR IGNORE INTO autores (nombre) VALUES (?)", (colaborador, ))
            con.commit()

            cur.execute("SELECT id_autor FROM autores WHERE nombre = ?", (colaborador, ))
            id_autor = cur.fetchone()[0]


        ##### Rellenando la tabla "isbn_autores" de la base de datos #####
            cur.execute("INSERT OR IGNORE INTO isbn_autores (id_isbn, id_autor, rol_autor) VALUES (?,?,?)", (id_isbn, id_autor, rol_autor, ))
            con.commit()



        ##### Rellenando la tabla "libros" de la base de datos #####
        # Hay que considerar que pueden existir dos libros distintos con el mismo titulo. Para tratar de minimizar estos casos, vamos a hacer un query que sea "titulo + colaboradores", y si esta combinacion ya se encuentra en la base de datos, el libro no se introduce
        cur.execute("SELECT titulo, colaboradores FROM libros WHERE titulo = ? AND colaboradores = ?", (titulo, colaboradores, ))
        coincidencia_sql = cur.fetchone()
        # print(coincidencia_sql)
        if coincidencia_sql is None:
            cur.execute("INSERT INTO libros (titulo, colaboradores) VALUES (?,?)", (titulo, colaboradores, ))
            con.commit()

        
        ##### Rellenando la tabla "isbn_libros" de la base de datos #####
        cur.execute("SELECT id_libro FROM libros WHERE titulo = ? AND colaboradores = ?", (titulo, colaboradores, ))
        id_libro = cur.fetchone()[0]
        cur.execute("INSERT OR IGNORE INTO isbn_libros (id_isbn, id_libro) VALUES (?,?)", (id_isbn, id_libro))
        con.commit()


        ##### Rellenando la tabla "libros_autores" de la base de datos #####   
        # Usamos el truco de antes: query que sea "titulo + colaboradores", y si esta combinacion ya se encuentra en la base de datos, el libro no se introduce. (No hace falta hacer otra vez el query: esta guardado en el objeto "coincidencia_sql").
        # Tambien hay que tener en cuenta las variables "colaboradores" que pueden presentar varios colaboradores separados con ";"
        if coincidencia_sql is None:
            #print(colaboradores)
            colaboradores_split = colaboradores.split(";")
            for colaborador in colaboradores_split:
                #print(colaborador)
                cur.execute("SELECT id_autor FROM autores WHERE nombre = ?", (colaborador, ))
                id_autor = cur.fetchone()[0]
                cur.execute("INSERT OR IGNORE INTO libros_autores (id_libro, id_autor) VALUES (?,?)", (id_libro, id_autor, ))
                con.commit()




        
    print()
    print("########## CARGA TERMINADA ##########")
    print(f"Excel '{item}' cargado en la base de datos. Pasando al siguiente excel...")
    print()
    print()
    print()
    print()


print()
print(f"Todos los archivos ({len(propia_files_done)}/{len(propia_files)}) de la carpeta han sido procesados: ")
for item in propia_files_done:
    print("          ", item.name)
        

#### Dataframe (scielo)
Esta información procede de **SciELO** (https://scielo.org/). Scielo se especializa en publicaciones electrónicas, así que la práctica totalidad de los libros tendrán formato ebook.
Del dataframe proporcionado podemos extraer información para rellenar algunas de las columnas de las tablas de la base de datos.

##### Información a limpiar:
- **Columna "publication_date"**:
    - Filtramos por los años 2020, 2021, 2022, 2024
- **Columnas "eisbn, isbn"**:
    - Una misma fila (es decir, un mismo libro) tiene tanto "eisbn" como "isbn". Hemos decidido que vamos a recoger el "eisbn" y, si el "eisbn" está ausente, cogemos el "isbn".
        - Además, si tiene "eisbn", en la columna de la base de datos "formato_publicacion" se pondrá el valor "ebook". Si no tiene "eisbn", en esta columna se pondrá "NULL"
- **Columna "publication_date"**:
    - Está en formato yyyy/mm/dd
- **Columna "creators"**:
    - Los valores de esta columna están divididos en items, de manera que 1 item = 1 autor.
        - Dentro de cada item hay 3 valores: "role", "full_name", "link_resume". Nos interesan los items "role" (para rellenar la tabla "isbn_autores") y "full_name".
            - En "full_name", el nombre está con formato "Primer apellido, Nombre" (por ejemplo, "[APELLIDO], [NOMBRE]"). Tenemos que darle la vuelta para que encaje con la lógica de la base de datos de "Nombre, Apellidos"
                - NOTA: hay inconsistencias: algunos casos son "Nombre Apellido", sin separación con comas. Ejemplo: [NOMBRE APELLIDO]
            - En "role", tenemos dos categorias:
                - organizer (lo traducimos como "organizador")
                - individual_author (lo traducimos como "autor")
- **Columna "country"**:
    - Está en formato de dos caracteres (ISO 3166-1 alfa-2). Por ejemplo: Brasil -> BR
        - **Añadir posteriormente un paso que decodifique esta información**
- **Columna "language"**:
    - Esta en formato de dos caracteres (ISO 639-1). Por ejemplo: español -> es; ingles -> en; portugués -> pt
        - **Añadir posteriormente un paso que decodifique esta información**
- **Columna "collection"**:
    - Los valores también están divididos en items. Sólo nos interesa el item "title" (que hace referencia al nombre de la colección)
- **Columna "doi_number"**:
    - Está en formato url. Nos interesa quedarnos solamente con el número.
- **Columna "shopping_info"**:
    - Los valores también están divididos en items. Cada item tiene dos valores: "store" y "book_url". Nos interesa el item "store"
        - El item "store" puede tener los valores "Amazon" o "Google Play", entre otros. 
            - **Añadir posteriormente: Si tiene "Amazon", lo renombraremos a "AMAZON.COM", si tiene "Google Play", lo renombraremos a "GOOGLE BOOKS". Hacemos esto para unificar los valores con el dataframe "ventas"**





In [ ]:
carpeta = Path("input_data/scielo")

scielo_files = list()

for item in carpeta.glob("*.xlsx"):
    if item.name.startswith("~$"):       # evita temporales de Excel
        continue
    print(item.name)
    scielo_files.append(item)

print()
print(f"{len(scielo_files)} excels a procesar...")
print()
print()
print()

In [ ]:
print("########## COMENZANDO LIMPIEZA ##########")
print()
count = 0

scielo_files_done = list()

for item in carpeta.glob("*.xlsx"):
    if item.name.startswith("~$"):       # evita temporales de Excel
        continue

    count = count + 1
    file_name = item
    print(f"{count}) Limpiando el excel: '{file_name}'")

    scielo = pd.read_excel(file_name, sheet_name=None)
    scielo = scielo["Sheet1"]

    # # Tratar las celdas que están vacías (incluido espacio en blanco) o que tienen NaN (el NA de pandas) a None de Python:
    # 1) Unificar las celdas vacías como valores faltantes
    scielo = scielo.replace(r'^\s*$', pd.NA, regex=True)
    # 2) Pasar todos los NA (NaN, NaT, pd.NA) a None de Python
    #    Importante: convertir antes a 'object' si quieres que se conserven los None
    scielo = scielo.astype("object").where(scielo.notna(), None)

    # Limpiamos la columna "publication_date"
    # Las filas tienen formato "yyyy-mm-dd". Pasarlas a "yyyy"
    # Normalizamos a string:
    scielo["publication_date"] = scielo["publication_date"].astype("string").str.strip()
    # Convertimos a datetime
    scielo["publication_date"] = pd.to_datetime(scielo["publication_date"], format="%Y-%m-%d")
    # Nos quedamos solo con el año
    scielo["publication_date"] = scielo["publication_date"].dt.year.astype("Int64") # Podriamos ponerlo directamente en "int", pero para curarnos en salud, ponemos "Int64" (por si en el futuro scielo nos envia libros que tienen NA en "publication_date")

    # Nos quedamos solo con las filas que tienen en la columna "publication_date" valores entre 2020 y 2024 (estos años incluidos)
    scielo = scielo[scielo["publication_date"].between(2020, 2024)]


    scielo_files_done.append(item)

    print("########## LIMPIEZA TERMINADA ##########")
    print(f"Excel '{item}' limpiado. Cargando los datos limpios a la base de datos...")

    scielo.to_excel("clean_scielo.xlsx", sheet_name="sheet1", index=False)

    print()
    print("########## CARGANDO A LA BASE DE DATOS ##########")

    for index, row in scielo.iterrows():
        id_original = row.loc["_id"]
        
        if row.loc["eisbn"] is not None:
            isbn13 = row.loc["eisbn"]
            formato_publicacion = "ebook"
            formato_comercial = "epub"
        else:
            isbn13 = row.loc["isbn"]
            formato_publicacion = None
            formato_comercial = None
        
        if isbn13 is None: # Si no tiene ni eisbn ni isbn, pasamos a la siguiente fila
            #print(id_original, isbn13)
            continue
        #if pd.isna(isbn13): print(id_original, isbn13)
        
        # Estas dos lineas de codigo es para testear que funciona bien el condicional de "if row.loc...". Lo puedes ignorar
        isbn13 = int(isbn13)
        #if isbn13.startswith("100"): print(isbn13)
        ## print("isbn: ", isbn13)

        titulo = row.loc["title"]
        anio_publicacion = row.loc["publication_date"]





        ###########################################
        ##### LIMPIANDO LA COLUMNA "CREATORS" #####
        # El valor de esta columna se divide en 3 items. Solo nos interesan los items "role" y "full_name"
        colaboradores = row.loc["creators"]
        ## print(colaboradores)

        # Dado que colaboradores es un string con esa estructura (ver el print), podemos convertirlo usando json después de “limpiar” un par de cosas:
        #   Cambiar None por null (para que sea JSON válido)
        #   Cambiar None por null (para que sea JSON válido)
        # Luego lo cargamos con json.loads y construimos los diccionarios con role y full_name



        colaboradores_list = list()

        # Si viene vacío o NaN, devolvemos lista vacía
        if isinstance(colaboradores, str) and colaboradores.strip():
            try:
                # Convierte el literal Python (con listas, '...' y None) a objeto Python
                colaboradores_raw = eval(colaboradores, {"__builtins__": {}}, {})
                # Ahora colaboradores_raw es algo como:
                # [[['role', 'organizer'], ['full_name', 'Backes, Carmen'], ...], [...], ...]
            except Exception as e:
                print("Error al evaluar colaboradores:", e)
                print(repr(colaboradores[:200]))
                colaboradores_raw = []
        else:
            colaboradores_raw = []

        # Mapeo de roles a español
        traducir_role = {
            "organizer": "organizador",
            "individual_author": "autor",
        }

        for pares in colaboradores_raw:
            datos = dict(pares)  # [['role', 'organizer'], ['full_name', '...'], ...] -> {'role': 'organizer', ...}
            role = datos.get("role")
            colaboradores_list.append({
                "role": traducir_role.get(role, role),
                "full_name": datos.get("full_name"),
            })








        ## print(colaboradores_list)

        # Vamos a darle la vuelta al nombre: vamos a ponerlo en formato "Nombre Apellido"
        new_colaboradores_list = list()

        for colaborador in colaboradores_list:
            rol = colaborador["role"]
            name = colaborador["full_name"]
            if re.findall(",", name):
                name_split = name.split(", ")
                #print(name_split)
                name = name_split[1].strip() + " " + name_split[0].strip()
                #print(name)

            new_colaboradores_list.append({
                "rol":rol,
                "name":name
            })
            
        ## print(new_colaboradores_list)



        # Para rellenar la columna "colaboradores" de la tabla "isbn" de la base de datos solo necesitamos el nombre del colaborador
        only_colaboradores = list()

        for new_colaborador in new_colaboradores_list:
            name = new_colaborador["name"]
            only_colaboradores.append(name)
        
        ## print(only_colaboradores)
        colaboradores = ";".join(only_colaboradores)
        ## print("colaboradores: ", colaboradores)
        ###########################################
        ###########################################



        pais_publicacion = row.loc["country"]
        editorial_principal_nombre = row.loc["publisher"]
        resumen = row.loc["synopsis"]
        idioma = row.loc["language"]



        #############################################
        ##### LIMPIANDO LA COLUMNA "COLLECTION" #####
        # El valor de esta columna se divide en 5 items. Solo nos interesa el item "title"
        coleccion = row.loc["collection"]
        ## print(coleccion)
        
        # Dado que coleccion es un string con esa estructura (ver el print), podemos convertirlo usando json después de “limpiar” un par de cosas:
        #   Cambiar None por null (para que sea JSON válido)
        #   Cambiar None por null (para que sea JSON válido)
        # Luego lo cargamos con json.loads y construimos los diccionarios con role y full_name

        coleccion = (
            coleccion.replace("None", "null").replace("'", '"')
        )

        coleccion_json = json.loads(coleccion)
        # print(json.dumps(coleccion_json, indent=8))

        title = dict(coleccion_json)

        coleccion = title["title"]
        ## print("collecion: ", coleccion)
        #############################################
        #############################################


        bisac = row.loc["bisac_code"]
        bisac = re.findall("code', '(.*)']]]", bisac)[0]

        tipo_obra = row.loc["TYPE"]
        
        doi_scielo = row.loc["doi_number"]
        doi_scielo = re.findall(".org/(.*)", doi_scielo)[0]




        ##### Rellenando la tabla "isbn" de la base de datos #####
        cur.execute("INSERT OR IGNORE INTO isbn (excel_origen, id_original, isbn13, formato_publicacion, titulo, anio_publicacion, colaboradores, pais_publicacion, ciudad_publicacion, editorial_principal_nombre, resumen, idioma, coleccion, bisac, tipo_obra, doi_scielo) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", (item.name, id_original, isbn13, formato_publicacion, titulo, anio_publicacion, colaboradores, pais_publicacion, ciudad_publicacion, editorial_principal_nombre, resumen, idioma, coleccion, bisac, tipo_obra, doi_scielo, ))
        con.commit()

        cur.execute("SELECT id_isbn FROM isbn WHERE isbn13 = ?", (isbn13, ))
        id_isbn = cur.fetchone()[0]
        # print(id_isbn)


        ##### Rellenando la tabla "editoriales" de la base de datos #####   
        cur.execute("INSERT OR IGNORE INTO editoriales (nombre) VALUES (?)", (editorial_principal_nombre, ))
        con.commit()

        cur.execute("SELECT id_editorial FROM editoriales WHERE nombre = ?", (editorial_principal_nombre, ))
        id_editorial = cur.fetchone()[0]


        ##### Rellenando la tabla "isbn_editoriales" de la base de datos #####
        rol_editorial = "editor"
        cur.execute("INSERT OR IGNORE INTO isbn_editoriales (id_isbn, id_editorial, rol_editorial) VALUES (?,?,?)", (id_isbn, id_editorial, rol_editorial, ))
        con.commit()




        ##### Rellenando la tabla "autores" de la base de datos #####
        ## print()
        ## print(new_colaboradores_list)
        ## print()

        for new_colaborador in new_colaboradores_list:
            #print(new_colaborador)
            rol_autor = new_colaborador["rol"]
            colaborador = new_colaborador["name"]
            #print(rol_autor, colaborador)

            cur.execute("INSERT OR IGNORE INTO autores (nombre) VALUES (?)", (colaborador, ))
            con.commit()

            cur.execute("SELECT id_autor FROM autores WHERE nombre = ?", (colaborador, ))
            id_autor = cur.fetchone()[0]


        ##### Rellenando la tabla "isbn_autores" de la base de datos #####
            cur.execute("INSERT OR IGNORE INTO isbn_autores (id_isbn, id_autor, rol_autor) VALUES (?,?,?)", (id_isbn, id_autor, rol_autor, ))
            con.commit()
            


        ##### Rellenando la tabla "libros" de la base de datos #####
        # Hay que considerar que pueden existir dos libros distintos con el mismo titulo. Para tratar de minimizar estos casos, vamos a hacer un query que sea "titulo + colaboradores", y si esta combinacion ya se encuentra en la base de datos, el libro no se introduce
        cur.execute("SELECT titulo, colaboradores FROM libros WHERE titulo = ? AND colaboradores = ?", (titulo, colaboradores, ))
        coincidencia_sql = cur.fetchone()
        # print(coincidencia_sql)
        if coincidencia_sql is None:
            cur.execute("INSERT INTO libros (titulo, colaboradores) VALUES (?,?)", (titulo, colaboradores, ))
            con.commit()


        ##### Rellenando la tabla "isbn_libros" de la base de datos #####
        cur.execute("SELECT id_libro FROM libros WHERE titulo = ? AND colaboradores = ?", (titulo, colaboradores, ))
        id_libro = cur.fetchone()[0]
        cur.execute("INSERT OR IGNORE INTO isbn_libros (id_isbn, id_libro) VALUES (?,?)", (id_isbn, id_libro))
        con.commit()


        ##### Rellenando la tabla "libros_autores" de la base de datos #####   
        # Usamos el truco de antes: query que sea "titulo + colaboradores", y si esta combinacion ya se encuentra en la base de datos, el libro no se introduce. (No hace falta hacer otra vez el query: esta guardado en el objeto "coincidencia_sql").
        # Tambien hay que tener en cuenta las variables "colaboradores" que pueden presentar varios colaboradores separados con ";"
        if coincidencia_sql is None:
            colaboradores_split = colaboradores.split(";")
            for colaborador in colaboradores_split:
                cur.execute("SELECT id_autor FROM autores WHERE nombre = ?", (colaborador, ))
                id_autor = cur.fetchone()[0]

                cur.execute("INSERT OR IGNORE INTO libros_autores (id_libro, id_autor) VALUES (?,?)", (id_libro, id_autor, ))
                con.commit()


        try:
            edicion = row.loc["edition"]
            ## print("edicion: ", edicion)
        except:
            edicion = None

        ################################################
        ##### LIMPIANDO LA COLUMNA "SHOPPING_INFO" #####
        # El valor de esta columna se divide en 3 items. Solo nos interesa el item "store"
        # Vamos a usar la misma logica de limpieza que usamos para la columna "COLLECTION"
        if not re.findall("store", row.loc["shopping_info"]):
            tienda = None
            ## print("tienda: ", tienda)
        else:
            tienda = row.loc["shopping_info"]
            #print(tienda)

            tienda = (
                tienda.replace("None", "null").replace("'", '"')
            )

            tienda_json = json.loads(tienda)
            #print(json.dumps(tienda_json, indent=8))

            ## print("tienda: ")
            for pares in tienda_json:
                valores = dict(pares)
                tienda = valores.get("store")
                ## print("   -", tienda)

        ##### Rellenando la tabla "ventas" de la base de datos #####
                # Aqui tenemos que un mismo libro puede venderse en varias tiendas, así que para evitar duplicados, limitamos por "id_isbn" + "tienda"
                cur.execute("SELECT * FROM ventas WHERE id_isbn = ? AND tienda = ? LIMIT 1", (id_isbn, tienda, ))
                row = cur.fetchone()
                if row is None:
                    cur.execute("INSERT INTO ventas (id_isbn, tienda, formato_comercial, edicion) VALUES (?,?,?,?)", (id_isbn, tienda, formato_comercial, edicion, ))
                    con.commit()

        ################################################
        ################################################



    print()
    print("########## CARGA TERMINADA ##########")
    print(f"Excel '{item}' cargado en la base de datos. Pasando al siguiente excel...")
    print()
    print()
    print()
    print()


print()
print(f"Todos los archivos ({len(scielo_files_done)}/{len(scielo_files)}) de la carpeta han sido procesados: ")
for item in scielo_files_done:
    print("          ", item.name)



   

#### Eliminando las filas que no tienen un ISBN13 real
Se han detectado algunas obras provenientes de los excel de SIMEH que en la columna "ISBN13" tienen un valor que **no** se corresponde con un ISBN13.
- ejemplo: titulo "[TÍTULO DE REGISTRO PRIVADO]", id_simeh "[ID_FUENTE]", ISBN13: "[ISBN_INVÁLIDO]"

Para no "contaminar" nuestra base de datos con estas obras que tienen un valor equivocado, las vamos a sacar de la base de datos y las recogeremos en un excel aparte que vamos a llamar ``obras_isbn13_erroneos.xlsx``

In [ ]:
# Detectar las obras que tienen ISBN13 falso

isbn13_erroneo = list()

for row in con.execute("SELECT id_isbn, isbn13 FROM isbn"):
    print(row)
    id_isbn = row[0]
    isbn13 = row[1]

    if len(isbn13) != 13 or not (isbn13.startswith("978") or isbn13.startswith("979")): 
        isbn13_erroneo.append(id_isbn)
        # print(id_isbn)
        # print(isbn13)
        # print()

# print(isbn13_erroneo)
placeholders = ",".join("?" * len(isbn13_erroneo))

query = f"SELECT * FROM isbn WHERE id_isbn IN ({placeholders})"
table = cur.execute(query, isbn13_erroneo).fetchall()
for row in table:
    print(row)

# Para exportar a excel una tabla de la base de datos sql
pd.read_sql_query(query, con, params=isbn13_erroneo).to_excel("obras_isbn13_erroneos.xlsx", index=False)


# Borrar las filas de la base de datos que tienen el isbn13 erroneo
query = f"DELETE FROM isbn WHERE id_isbn IN ({placeholders})"
cur.execute(query, isbn13_erroneo)
con.commit()
print(f"Eliminadas {cur.rowcount} filas de la tabla isbn.")


#### Dataframe (altmetric)
Esta información procede de **Altmetric** (https://altmetric.com). Ofrece información relativa a las métricas alternativas de las publicaciones.

In [ ]:
# Exportando una lista de isbns para sacar las metricas de Altmetrics.com
isbn13_list = list()
with open("isbn13_list.txt", "w", encoding="utf-8") as file:
    for row in con.execute("SELECT isbn13 FROM isbn"):
        isbn13 = row[0]
        if len(isbn13) != 13: continue
        if not isbn13.startswith("97"): continue
        isbn13_list.append(isbn13)
        # print(len(isbn13))
        file.write(f"{isbn13}\n")
print()
print(len(isbn13_list))

In [ ]:
# Limpiando el output (archivo .csv) de altmetrics

carpeta = Path("input_data/altmetrics")

altmetrics_files = list()

for item in carpeta.glob("*.csv"):
    if item.name.startswith("~$"):       # evita temporales de Excel
        continue
    print(item.name)
    altmetrics_files.append(item)

print()
print(f"{len(altmetrics_files)} archivos a procesar...")


In [ ]:
altmetrics = pd.read_csv(item, encoding="cp1252")
# altmetrics.to_excel("altmetrics.xlsx", index=False)
altmetrics

In [ ]:
# Convertir la columna "isbn" a integer
altmetrics["ISBN"] = altmetrics["ISBN"].astype("Int64") # Usamos "Int64" aplicando la misma lógica de antes (mientras limpiábamos el dataframe "simeh"):  ya que esta columna tiene valores NA (tipo pd.NA) y "astype(int)" no admite NAtype (te daria error si usas "astype(int)"). Esta alternativa "astype("Int64")" permite una variante de entero que admite faltantes (pd.NA). No es lo mismo que el int64 de NumPy.

In [ ]:
for index, row in altmetrics.iterrows():
    #print(row)
    isbn = row.loc["ISBN"]
    print(isbn)

    if pd.isna(isbn): # Como los NAs del dataframe son tipo "pd.NA" y no son tipo "None", usamos "pd.isna()". Si fueran "None" podriamos usar "if isbn is None: continue"
        continue

    news_mentions = row.loc["News mentions"]
    blog_mentions = row.loc["Blog mentions"]
    policy_mentions = row.loc["Policy mentions"]
    patents_mentions = row.loc["Patent mentions"]
    x_mentions = row.loc["X mentions"]
    peer_reviews_mentions = row.loc["Peer review mentions"]
    weibo_mentions = row.loc["Weibo mentions"]
    facebook_mentions = row.loc["Facebook mentions"]
    wikipedia_mentions = row.loc["Wikipedia mentions"]
    google_plus_mentions = row.loc["Google+ mentions"]
    linkedin_mentions = row.loc["LinkedIn mentions"]
    reddit_mentions = row.loc["Reddit mentions"]
    pinterest_mentions = row.loc["Pinterest mentions"]
    f1000_mentions = row.loc["F1000 mentions"]
    q_a_mentions = row.loc["Q&A mentions"]
    video_mentions = row.loc["Video mentions"]
    clinical_guidelines_mentions = row.loc["Clinical guidelines mentions"]
    bluesky_mentions = row.loc["Bluesky mentions"]
    podcast_mentions = row.loc["Podcast mentions"]
    syllabi_mentions = row.loc["Syllabi mentions"]
    mendeley_readers = row.loc["Number of Mendeley readers"]
    dimensions_citations = row.loc["Number of Dimensions citations"]

    cur.execute("UPDATE isbn SET news_mentions = ?, blog_mentions = ?, policy_mentions = ?, patents_mentions = ?, x_mentions = ?, peer_reviews_mentions = ?, weibo_mentions = ?, facebook_mentions = ?, wikipedia_mentions = ?, google_plus_mentions = ?, linkedin_mentions = ?, reddit_mentions = ?, pinterest_mentions = ?, f1000_mentions = ?, q_a_mentions = ?, video_mentions = ?, clinical_guidelines_mentions = ?, bluesky_mentions = ?, podcast_mentions = ?, syllabi_mentions = ?, mendeley_readers = ?, dimensions_citations = ? WHERE isbn13 = ?", (news_mentions, blog_mentions, policy_mentions, patents_mentions, x_mentions, peer_reviews_mentions, weibo_mentions, facebook_mentions, wikipedia_mentions, google_plus_mentions, linkedin_mentions, reddit_mentions, pinterest_mentions, f1000_mentions, q_a_mentions, video_mentions, clinical_guidelines_mentions, bluesky_mentions, podcast_mentions, syllabi_mentions, mendeley_readers, dimensions_citations, isbn, ))
    con.commit()    

#### Dataframe (libreria siglo - ventas)
La información procede de la **Libreria Siglo** (https://libreriasiglo.com/). Ofrece información sobre las ventas (por ISBN).

##### Información a limpiar:
- La única información que supone cierta complejidad a la hora de limpiar y preparar para introducirla en nuestra base de datos es confirmar si el Excel hace referencia a las ventas de un año natural (del 1 de enero al 31 de diciembre) y extraer el año de venta. Por suerte, parece que la plantilla es consistente y siempre muestra un texto con el formato ``Desde: 1 de enero de 2024 Hasta: 31 de diciembre de 2024``, por lo que podemos parsear este texto para extraer la información mencionada.
- Las 6 primeras filas del Excel están en blanco. Aunque parece que esto también es consistente, introduciremos por si acaso un algoritmo lo suficientemente inteligente como para identificar automáticamente cuándo empieza la fila de interés y borrar las columnas en blanco.

- **Columna "Cantidad"**
    - Se ha detectado una errata en esta columna: hay un "-1" (ver ISBN: [ISBN_PRIVADO]). Los números negativos los vamos a cambiar por 0

In [ ]:
carpeta = Path("input_data/ventas")

ventas_files = list()

for item in carpeta.glob("*.xlsx"):
    if item.name.startswith("~$"):       # evita temporales de Excel
        continue
    print(item.name)
    ventas_files.append(item)

print()
print(f"{len(ventas_files)} excels a procesar...")
print()
print()
print()

In [ ]:
print("########## COMENZANDO LIMPIEZA ##########")
print()
count = 0

ventas_files_done = list()
files_no_anio_natural = list()

for item in carpeta.glob("*.xlsx"):
    if item.name.startswith("~$"):       # evita temporales de Excel
        continue

    count = count + 1
    file_name = item
    print(f"{count}) Limpiando el excel: '{file_name}'")

    ventas = pd.read_excel(file_name, sheet_name=None)
    ventas = ventas["VentasxFulldata"]


# Primero vamos a asegurarnos de que la información de ventas que vamos a meter corresponde a un año natural completo (i.e., 1 de enero XXXX - 31 de diciembre XXXX)

    anio_natural = False

    for index, row in ventas.iterrows():
        row = row.tolist()
        for item in row:
            # si la celda no es texto (NaN, número, None...), la saltamos
            if not isinstance(item, str):
                continue

            # buscar las dos frases básicas
            if "Desde" in item and "Hasta" in item:
                fecha = item

                # extraemos el año de la parte de "31 de diciembre de XXXX"
                anio = re.findall(r"31 de diciembre de (\d{4})", fecha)
                if not anio:
                    # no hay año claro, pasamos a la siguiente celda
                    continue

                anio = anio[0]

                # comprobamos que las dos fechas son del MISMO año
                if f"1 de enero de {anio}" in fecha and f"31 de diciembre de {anio}" in fecha:
                    anio_venta = anio
                    # print(anio_venta)
                    anio_natural = True
                else:
                    pass

    if anio_natural is True:
        print(f"### SI es el año natural: {fecha} ###")
    else:
        print(f"### NO es el año natural: {fecha} ###")
        files_no_anio_natural.append(file_name)
        continue
        

    # Aunque parece que en el excel la fila de interés comienza en la fila numero 5 del dataframe (7 en el excel), vamos a hacer un algoritmo que detecte donde empieza la fila de interés, por si recibimos algún documento en el que la fila de interés empiece en otro lugar.
    # La fila de interes debe tener los valores: "Canal", "Tienda", "Modalidad", "País", "Área", "Subárea", "Código", "Formato", "Isbn", "Título", "Edición", "Autor", "Cantidad"
    fila_interes = ["Canal", "Tienda", "Modalidad", "País", "Área", "Subárea", "Código", "Formato", "Isbn", "Título", "Edición", "Autor", "Cantidad"]
    fila_interes_v2 = ["Canal", "Tienda", "País", "Área", "Subárea", "Modalidad", "Código", "Formato", "Isbn", "Título", "Edición", "Autor", "Cantidad"] # Esta v2 es para los excel nuevos llamados "Ventas ebook 20XX.xlsx" que tienen un orden distinto de columnas

    count_rows = 0
    for index, row in ventas.iterrows():
        count_rows = count_rows + 1
        row = row.tolist()
        if row == fila_interes or row == fila_interes_v2:
            print(row)
            print(count_rows)
            break
    # Borramos las columnas que no son de interés
    ventas = ventas.iloc[count_rows:]

    # Renombramos las columas
    if row == fila_interes: ventas.columns = fila_interes
    if row == fila_interes_v2: ventas.columns = fila_interes_v2


    ventas_files_done.append(file_name)

    print()
    print("########## LIMPIEZA TERMINADA ##########")
    print(f"Excel '{file_name}' limpiado. Cargando los datos limpios a la base de datos...")




    print()
    print("########## CARGANDO A LA BASE DE DATOS ##########")

    for index, row in ventas.iterrows():
        # print(row)
        isbn = row.loc["Isbn"]
        canal = row.loc["Canal"]
        tienda = row.loc["Tienda"]
        modalidad = row.loc["Modalidad"]
        pais_venta = row.loc["País"]
        area = row.loc["Área"]
        subarea = row.loc["Subárea"]
        formato_comercial = row.loc["Formato"]
        edicion = row.loc["Edición"]
        cantidad_vendida = row.loc["Cantidad"]

        print(isbn, canal, tienda, modalidad, pais_venta, area, subarea, formato_comercial, edicion, cantidad_vendida)

        cur.execute("SELECT id_isbn FROM isbn WHERE isbn13 = ?", (isbn, ))
        id_isbn = cur.fetchone()
        if id_isbn is None:
            continue
        id_isbn = id_isbn[0]
        # print(id_isbn)

        ##### Rellenando la tabla "ventas" de la base de datos #####
        # Primero, vamos a evitar meter la misma info dos veces: si coinciden el id_isbn, el año de venta, el canal, la tienda, la modalidad, y el pais, pasamos a la siguiente fila:
        cur.execute("SELECT * FROM ventas WHERE id_isbn = ? AND anio_venta = ? AND canal = ? AND tienda = ? AND modalidad = ? AND pais_venta = ? LIMIT 1", (id_isbn, anio_venta, canal, tienda, modalidad, pais_venta, ))
        row = cur.fetchone()
        if row is None:
            cur.execute("INSERT INTO ventas (id_isbn, anio_venta, canal, tienda, modalidad, pais_venta, area, subarea, formato_comercial, edicion, cantidad_vendida) VALUES (?,?,?,?,?,?,?,?,?,?, ?)", (id_isbn, anio_venta, canal, tienda, modalidad, pais_venta, area, subarea, formato_comercial, edicion, cantidad_vendida, ))
            con.commit()

        
    print()
    print("########## CARGA TERMINADA ##########")
    print(f"Excel '{file_name}' cargado en la base de datos. Pasando al siguiente excel...")
    print()
    print()
    print()
    print()



print()
print(f"Todos los archivos ({len(ventas_files_done)}/{len(ventas_files)}) de la carpeta han sido procesados: ")
for item in ventas_files_done:
    print("          ", item.name)



if len(files_no_anio_natural) != 0:
    print()
    print("Archivos que no han sido procesados por no tener información relativa a un año natural: ")
    for item in files_no_anio_natural:
            print("          ", item.name)
    


### 3) Limpiando la base de datos

En esta parte del script vamos a limpiar las tablas de nuestra base de datos. Con limpiar nos referimos a estandarizar los valores de las columnas, ya que como la información de nuestra base de datos procede de distintas instituciones (cada una con sus respectivas normas a la hora de organizar sus datos) nos encontramos con que una columna puede tener la misma información escrita de distinta manera, lo que ocasionaría ruido a la hora de analizar los datos.

También hay que corregir algunas erratas procedentes de las instituciones.

Columnas a limpiar:

- Tabla **isbn**
    - Columna **pais_publicacion**:
        - Los datos provenientes de simeh se refieren al país por su nombre completo, e.g: *España, Brasil, Colombia, Ecuador*
        - Los datos provenientes de scielo se refieren al país con dos caracteres, siguiendo la normativa *ISO 3166-1* (https://es.wikipedia.org/wiki/ISO_3166-1_alfa-2#AA). Utilizando los ejemplos de antes, tendríamos: *ES, BR, CO, EC*
        - Aunque creo que lo correcto sería pasarlo todo a la normativa ISO, para que sea más legible vamos a pasarlo todo al formato de nombre completo.

    - Columna **ciudad_publicación**:
        - Similar al punto anterior, una misma ciudad puede estar escrita de diferente manera. Por ejemplo: "Ciudad de México / cdmx / CDMX"

    - Columna **idioma**:
        - Los datos provenientes de simeh a veces se refieren al "español" como "*Spanish*" y otras veces como "*Español / Castellano*". Otros ejemplos que nos encontramos son "*Inglés*", "*Portugués*", etc. Es decir, simeh codifica el idioma utilizando el nombre completo.
        - Los datos provenientes de scielo se refieren al idioma con dos caracteres, siguiendo la normativa *ISO 639-1* (https://es.wikipedia.org/wiki/ISO_639-1). Utilizando los ejemplos de antes, tendríamos: *es, en, pt*
        - Aunque creo que lo correcto sería pasarlo todo a la normativa ISO, para que sea más legible vamos a pasarlo todo al formato de nombre completo.

    - Columna **formato_publicacion**:
        - Cambiar *print* por *impreso*
        
    - Columna **tipo_obra**:
        - Cambiar *Monograph* por *monografia*

    - Columna **tipo_coedicion**:
        - Los datos que provienen de simeh y de la plantilla_propia si tienen estas variables (es decir, tendrán "si/no"), pero los datos provenientes de scielo no (es decir, son "None"). Esto quiere decir que en la base de datos habrá tres tipos de valores: si/no/None. Tenemos que tomar una decisión sobre como tratar los "None"

    - Columna **colaboradores**:
        - Algunas editoriales optan por poner "Varios autores", "Varios Autores", "autores Varios" en vez de los nombres de los colaboradores.
        - El que tenga la "a" mayúscula o minúscula puede ocasionar problemas, ya que la base de datos lo interpreta como un valor distinto, así que vamos a reemplazar "Varios Autores" y "autores Varios" por "Varios autores"
        
- Tabla **ventas**
    - Columna **tienda**:
        - Ejemplo de datos provenientes de la libreria siglo: *AMAZON.COM, GOOGLE BOOKS, KOBO.COM*
        - Ejemplo de datos provenientes de scielo: *Amazon, Google Play, Kobo Books*
        - Usaremos el formato de libreria siglo

    - Columna **cantidad_vendida**:
        - Corregir errata **-1** (id_isbn = [ID_INTERNO]; isbn13 = [ISBN_PRIVADO]). Esta errata procede de la libreria siglo (concretamente del excel "*[INFORME_VENTAS_PRIVADO]*")
        - Cambiaremos los **-1** a a **None**. 

- Tabla **autores**:
    - Columna **nombre**:
        - Algunas editoriales optan por poner "Varios autores", "Varios Autores", "autores Varios" en vez de los nombres de los colaboradores.
        - El que tenga la "a" mayúscula o minúscula puede ocasionar problemas, ya que la base de datos lo interpreta como un valor distinto, así que vamos a reemplazar "Varios Autores" y "autores Varios" por "Varios autores". **PROBLEMA: esta columna tiene el atributo "UNICO", asi que hace resto da problemas. Hay que mirar otra forma**

- Tabla **isbn_autores**:
    - Columna **rol_autor**:
        - Cambiar *translator* por *traductor*
        - Cambiar *organizador* por *compilador*


In [ ]:
##################################
#### LIMPIANDO LA TABLA VENTAS ###

# Pasando los -1 de la columna "cantidad_vendida" a None:
cur.execute("UPDATE ventas SET cantidad_vendida = NULL WHERE cantidad_vendida = -1")

# Estandarizando el nombre de las tiendas
cur.executescript("""
UPDATE ventas SET tienda = "AMAZON.COM" WHERE tienda = "Amazon";
UPDATE ventas SET tienda = "GOOGLE BOOKS" WHERE tienda = "Google Books";
UPDATE ventas SET tienda = "GOOGLE PLAY" WHERE tienda = "Google Play";
UPDATE ventas SET tienda = "KOBO.COM" WHERE tienda = "Kobo Books"
""")
con.commit()


In [ ]:
##################################
#### LIMPIANDO LA TABLA ISBN ###

# Estandarizando el nombre de los paises:
# Por ahora, vamos a limpiarla a mano. En el futuro no habría problema en automatizar el proceso.
# Primero, vamos a ver los valores unicos de la columna "pais_publicacion"

query = """
SELECT DISTINCT pais_publicacion
FROM isbn
ORDER BY pais_publicacion;
"""
for row in cur.execute(query):
    print(row)

# BR lo renombraremos a "Brasil"
cur.execute("UPDATE isbn SET pais_publicacion = 'Brasil' WHERE pais_publicacion = 'BR'")
cur.execute("UPDATE isbn SET pais_publicacion = 'Perú' WHERE pais_publicacion = 'Peru'")
con.commit()

In [ ]:
cur.execute("""
    SELECT DISTINCT pais_publicacion
    FROM isbn
    ORDER BY pais_publicacion
""")
paises = [fila[0] for fila in cur.fetchall()]
print(paises)


In [ ]:
# HAY QUE LIMPIAR EL NOMBRE DE LAS CIUDADES #
query = """
SELECT DISTINCT ciudad_publicacion
FROM isbn
ORDER BY ciudad_publicacion;
"""
for row in cur.execute(query):
    print(row)

In [ ]:
# Estandarizando las categorias de idiomas
query = ("""
SELECT DISTINCT idioma
FROM isbn
ORDER BY idioma
""")

for row in cur.execute(query):
    print(row)

cur.executescript("""
-- 1) Cambiar cualquier 'Español / Castellano' (solo o dentro de otro string) a 'Español'
UPDATE isbn SET idioma = REPLACE(idioma, 'Español / Castellano', 'Español') WHERE idioma LIKE '%Español / Castellano%';

-- 2) Unificar 'Spanish' y 'es' a 'Español'
UPDATE isbn SET idioma = 'Español' WHERE idioma IN ('Spanish', 'es');

-- 3) Reordenar 'Inglés;Español' a 'Español;Inglés'
UPDATE isbn SET idioma = 'Español;Inglés' WHERE idioma = 'Inglés;Español';
                  
-- 3) Cambiar 'en' por 'Inglés'
UPDATE isbn SET idioma = 'Inglés' WHERE idioma = 'en';
                  
-- 4) Cambiar 'pt' por "Portugués"
UPDATE isbn SET idioma = 'Portugués' WHERE idioma = 'pt';

-- 5) Cambiar 'Lenguas sin código específico' por 'No determinado'
UPDATE isbn SET idioma = 'No determinado' WHERE idioma = 'Lenguas sin código específico';
                  
-- 6) Cambiar 'ita' por "Italiano"
UPDATE isbn SET idioma = 'Italiano' WHERE idioma = 'ita'            
                  
""")

con.commit()


In [ ]:
query = ("""
SELECT DISTINCT idioma
FROM isbn
ORDER BY idioma
""")

for row in cur.execute(query):
    print(row)



In [ ]:
# Cambiando "print" por "impreso"
query = ("""
SELECT DISTINCT formato_publicacion
FROM isbn
ORDER BY formato_publicacion
""")

for row in cur.execute(query):
    print(row)

cur.executescript("""
-- 1) Cambiar cualquier 'print' (solo o dentro de otro string) a 'impreso'
UPDATE isbn SET formato_publicacion = REPLACE(formato_publicacion, 'print', 'impreso') WHERE formato_publicacion LIKE '%print%';  
""")

con.commit()


In [ ]:
query = ("""
SELECT DISTINCT formato_publicacion
FROM isbn
ORDER BY formato_publicacion
""")

for row in cur.execute(query):
    print(row)


In [ ]:
# Cambiando "Monograph" y otras variantes por "monografia";
query = ("""
SELECT DISTINCT tipo_obra
FROM isbn
ORDER BY tipo_obra
""")

for row in cur.execute(query):
    print(row)

cur.executescript("""
UPDATE isbn
SET tipo_obra = 'monografia'
WHERE tipo_obra IN ('Monograph', 'Monográfica', 'Monográfico');
""")

con.commit()

In [ ]:
query = ("""
SELECT DISTINCT tipo_obra
FROM isbn
ORDER BY tipo_obra
""")

for row in cur.execute(query):
    print(row)

In [ ]:
# Reemplazando "Varios Autores" y "autores Varios" por "Varios autores"
# cur.executescript("""
# UPDATE isbn SET colaboradores = REPLACE(colaboradores, 'Varios Autores', 'Varios autores') WHERE colaboradores LIKE '%Varios Autores%';
# UPDATE isbn SET colaboradores = REPLACE(colaboradores, 'autores Varios', 'Varios autores') WHERE colaboradores LIKE '%autores Varios%';  
# """)
# con.commit()

In [ ]:
##################################
#### LIMPIANDO LA TABLA ISBN_AUTORES ###
# Cambiando "translator" por "traductor" y "organizador" por "compilador"
query = ("""
SELECT DISTINCT rol_autor
FROM isbn_autores
ORDER BY rol_autor
""")

for row in cur.execute(query):
    print(row)
try:
    cur.executescript("""
    UPDATE isbn_autores SET rol_autor = REPLACE(rol_autor, 'translator', 'traductor') WHERE rol_autor LIKE '%translator%';  
    UPDATE isbn_autores SET rol_autor = REPLACE(rol_autor, 'organizador', 'compilador') WHERE rol_autor LIKE '%organizador%';  
    """)
    con.commit()
except:
    pass

In [ ]:
query = ("""
SELECT DISTINCT rol_autor
FROM isbn_autores
ORDER BY rol_autor
""")

for row in cur.execute(query):
    print(row)

In [ ]:
# Test: hacer consulta SQL para asegurarnos de que funciona bien

### 4) APIs
En esta parte del script vamos a rellenar la base de datos utilizando distintas APIs

#### OpenAlex API
En esta parte del script, vamos a rellenar la columna "citacion" y "doi_openalex" de la tabla "isbn" usando la API de OpenAlex. 
OpenAlex no acepta búsquedas por ISBN, así que vamos a tener que hacer la búsqueda por el título del libro.
El principal reto al que nos enfrentamos es producir un algoritmo que sea lo suficientemente "inteligente" para que elija el libro correcto entre los resultados de búsqueda de OpenAlex (para los casos en los que el resultado de búsqueda arroje más de 1 resultado).
Para resolver esto, usaremos la técnica del "fuzzy matching": 
- **1)** iremos fila por fila recogemos el título del libro y los colaboradores de nuestra base de datos, y normalizamos los valores (i.e., ponemos todo en minuscula, sin acentos); 
- **2)** acto seguido, recogemos en una lista los distintos resultados arrojados por OpenAlex, y esta lista la dividimos, por un lado, en titulo del libro, y por otro, en colaboradores.
- **3)** comparamos la similitud entre el titulo del libro de nuestra base de datos con los distintos titulos que han resultado de la búsqueda de OpenAlex, y comparamos también la similitud entre el apellido de los colaboradores del libro de nuestra base de datos con los colaboradores asociados a los distintos titulos que han resultado de OpenAlex.
- **4)** Nos quedamos con el resultado que tenga una mayor similitud con el título y apellidos de colaboradores de nuestra base de datos (esto es el "fuzzy matching")

Podemos **ponderar la importancia** del match entre títulos y entre apellidos: podemos darle un 80% de importancia al título, y un 20% a los apellidos.

##### Limitaciones:
- He observado que tenemos en la base de datos libros que sí están presentes en OpenAlex pero no con el título completo, y esto puede dar falsos negativos en OpenAlex. Por ejemplo, en la base de datos tenemos un libro titulado "[TÍTULO PRINCIPAL]: [SUBTÍTULO]". Si buscamos tal cual en OpenAlex, nos sale que este libro no se encuentra en la base de datos de OpenAlex. Sin embargo, si buscamos por "[TÍTULO PRINCIPAL]", si lo encontramos.
    - Para tratar de solucionar este problema, se me ha ocurrido buscar en OpenAlex utilizando la parte del título anterior a ":" o ".". El inconveniente es que libros titulados, por ejemplo, "[TÍTULO COMPARTIDO]: Volumen 1", "[TÍTULO COMPARTIDO]: Volumen 2", etc., y escritos por el mismo autor o autores, se identificarán en nuestra base de datos como un mismo libro (cuando no lo son), por lo que se les asignarán las mismas citas y el mismo DOI. 
        - Posible solución (a futuro): filtrar los libros que tengan el mismo "doi_openalex" pero distinto titulo, y corregirlos manualmente.

            Consulta SQL para hacer este filtro:

                    WITH x AS (
                    SELECT DISTINCT
                        l.id_libro,
                        l.titulo,
                        TRIM(i.doi_openalex) AS doi_openalex
                    FROM libros AS l
                    JOIN isbn_libros AS il ON il.id_libro = l.id_libro
                    JOIN isbn        AS i  ON i.id_isbn  = il.id_isbn
                    WHERE COALESCE(TRIM(i.doi_openalex), '') <> ''
                    ),
                    dupes AS (
                    SELECT doi_openalex
                    FROM x
                    GROUP BY doi_openalex
                    HAVING COUNT(DISTINCT titulo) > 1
                    )
                    SELECT
                    x.doi_openalex,
                    x.id_libro,
                    x.titulo
                    FROM x
                    JOIN dupes d ON d.doi_openalex = x.doi_openalex
                    ORDER BY x.doi_openalex, x.titulo, x.id_libro;


In [ ]:
# ------------------------------
# Normalizar (minúsculas + sin tildes + strip)
# ------------------------------

# Creamos una funcion que nos permitirá normalizar los títulos de los libros y los nombres de los autores (con normalizar nos referimos a: poner todo en minusculas, quitar acentos, quitar espacios al principio y final del string)

def norm(s: str) -> str:
    """strip + lower + quita tildes/diacríticos (NFD)."""
    if s is None:
        return ""
    s = s.strip().lower()
    s = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

# ------------------------------
# Fuzzy matching (simple y legible)
# ------------------------------
def token_set_ratio(a: str, b: str) -> float:
    """
    Similaridad 0..100 robusta al orden y a palabras extra:
    - Tokeniza en palabras (conjuntos)
    - Compara intersección vs (intersección + resto) en ambos sentidos
    - Toma el máximo
    """
    aset, bset = set(a.split()), set(b.split())
    inter = " ".join(sorted(aset & bset))
    a_only = " ".join(sorted(aset - bset))
    b_only = " ".join(sorted(bset - aset))

    r = lambda x, y: SequenceMatcher(None, x, y).ratio()
    score1 = r(inter, (inter + " " + a_only).strip())
    score2 = r(inter, (inter + " " + b_only).strip())
    return max(score1, score2) * 100.0

def split_names(s: str):
    """Convierte 'A;B;C' en lista ['A','B','C'] (ya normalizados previamente)."""
    if not s:
        return []
    return [p for p in s.split(";") if p]

def author_score_fuzzy(colabs_db: str, colabs_oa: str, name_threshold: float = 90.0) -> float:
    """
    Empareja nombres de tu lista (DB) con la de OpenAlex (greedy, 1 a 1).
    Cuenta como match si token_set_ratio(nombre_db, nombre_oa) >= name_threshold.
    Devuelve proporción de nombres de DB que encontraron match (0..1).
    """
    q = split_names(colabs_db)
    c = split_names(colabs_oa)
    if not q or not c:
        return 0.0

    used = [False] * len(c)
    matches = 0

    for qn in q:
        best_i, best_ratio = -1, -1.0
        for i, cn in enumerate(c):
            if used[i]:
                continue
            r = token_set_ratio(qn, cn)
            if r > best_ratio:
                best_ratio, best_i = r, i
        if best_ratio >= name_threshold and best_i >= 0:
            used[best_i] = True
            matches += 1

    return matches / len(q)

def title_score_fuzzy(q_title: str, cand_title: str) -> float:
    """
    Puntuación de título:
    - 1.0 si exacto
    - 0.8 si token_set_ratio >= 90
    - 0.6 si token_set_ratio >= 80
    - 0.0 en caso contrario
    """
    if q_title and q_title == cand_title:
        return 1.0
    r = token_set_ratio(q_title, cand_title)
    if r >= 90:
        return 0.8
    elif r >= 80:
        return 0.6
    return 0.0



# ==============================
# LOOP
# ==============================

# Vamos a usar la tabla "libros" para sacar los títulos y colaboradores de nuestra base de datos
for row in con.execute("SELECT * FROM libros"):
    id_libro = row[0]
    titulo = row[1] # Nos interesa conservar el titulo tal cual sale del query para usarlo mas adelante
    titulo_norm = titulo.split(":")[0] # Nos quedamos con el titulo principal (lo que hay antes de ":". Por ejemplo, si tenemos el libro "Cambio climatico: impactos y consecuencias", nos quedamos solo con la parte del titulo "Cambio climatico")
    titulo_norm = titulo_norm.split(".")[0] # A veces el titulo y el subtitulo vienen separado con ".", así que lo incluimos tambien.

    titulo_norm = titulo_norm.strip().lower() # Hemos visto que OpenAlex es sensible a los acentos... por ejemplo, si tenemos el título de libro "[TÍTULO CON ACENTOS]", si buscamos la versión sin acentos ("[TÍTULO SIN ACENTOS]"), OpenAlex no encuentra nada.
    colaboradores = row[2] # Nos interesa conservar los colaboradores tal cual sale del query para usarlo mas adelante
    colaboradores_norm = norm(row[2])

    # cur.execute("SELECT citacion FROM isbn WHERE titulo = ? AND colaboradores = ? LIMIT 1", (titulo, colaboradores, )) # Esto lo hacemos para agilizar el proceso y que el script continue donde lo dejo
    # row = cur.fetchone()
    # if row is not None: 
    #     citacion = row[0]
    #     if citacion is not None:
    #         continue

    cur.execute("SELECT openalex_consulted FROM isbn WHERE titulo = ? AND colaboradores = ? LIMIT 1", (titulo, colaboradores, )) # Esto lo hacemos para agilizar el proceso y que el script continue donde lo dejo
    row = cur.fetchone()
    print("openalex_consulted:",row)
    if row is not None:
        openalex_consulted = row[0]
        if openalex_consulted is not None:  # Si el libro en cuestion ya ha sido consultado en OpenAlex, pasamos al siguiente
            print()
            continue

    print(" ########## Base de datos ########## ")
    print("titulo: ",titulo_norm)
    print("colaboradores: ", colaboradores_norm)
    print()

    # Conexion con OpenAlex
    per_page = 200
    filter = "type:book"

    url = "https://api.openalex.org/works"
    params = {"search":titulo_norm, "per_page":per_page, "filter":filter}

    response = requests.get(url, params=params)
    data = response.json()

    ## print(json.dumps(data, indent=8, ensure_ascii=False))
    ## print()
    try:
        if data["meta"]["count"] == 0:
            cur.execute("UPDATE isbn SET openalex_consulted = ? WHERE titulo = ? AND colaboradores = ? ", ("not found", titulo, colaboradores, ))
            con.commit()
            continue # Si la API devuelve 0 resultados, pasamos a la siguiente iteracion (i.e., siguiente titulo de libro)
    except:
        cur.execute("UPDATE isbn SET openalex_consulted = ? WHERE titulo = ? AND colaboradores = ? ", ("not found", titulo, colaboradores, ))
        con.commit()
        continue
    
    print(" ########## OpenAlex ########## ")
    libros_openalex = list()
    for item in data["results"]:
        print("item_titulo: ", item["title"])
        try:
            item_titulo = item["title"].strip().lower() # normalizamos los titulos del dataframe que hemos sacado de OpenAlex
            # item_titulo = item_titulo[:len(titulo_norm)] # Esto es opcional, pero quizás hace que mejore la precisión del match
        except:
            cur.execute("UPDATE isbn SET openalex_consulted = ? WHERE titulo = ? AND colaboradores = ? ", ("except (title)", titulo, colaboradores, ))
            con.commit()
            continue
        item_id = item["id"]
        try:
            doi_openalex = item["doi"]
            doi_openalex = re.findall(".org/(.+)", doi_openalex)[0] # nos quedamos con el codigo
        except:
            doi_openalex = None

        ## print(doi_openalex)
        print("titulo: ", item_titulo)

        item_colaboradores = item["authorships"]
        colaboradores_openalex = list()
        for item in item_colaboradores:
            
            author = item["author"]["display_name"]
            author = norm(author) # normalizamos el nombre de los colaboradores del dataframe que hemos sacado de OpenAlex
            colaboradores_openalex.append(author)

        print("colaboradores: ", colaboradores_openalex)

        libros_openalex.append({
        "titulo_openalex":item_titulo,
        "colaboradores_openalex":";".join(colaboradores_openalex),
        "id_openalex":item_id,
        "doi_openalex":doi_openalex
        })
        
    libros_openalex_df = pd.DataFrame(libros_openalex)

    # ==============================
    # Calcular puntuaciones y ordenar
    # ==============================
    
    filas_scored = list()
    for index, row in libros_openalex_df.iterrows():

        titulo_openalex = row["titulo_openalex"]
        colaboradores_openalex = row["colaboradores_openalex"]
        id_openalex = row["id_openalex"]
        doi_openalex = row["doi_openalex"]

        titulo_score = title_score_fuzzy(titulo_norm, titulo_openalex)
        colaborador_score = author_score_fuzzy(colaboradores_norm, colaboradores_openalex, name_threshold=90.0)

        if colaborador_score == 0: # Esto lo hacemos para la siguiente logica: forzamos que, aunque el título sea perfecto (titulo_score), si no hay match de autores (colaborador_score), el final_score sea 0. Esto lo hacemos para evitar que entren libros que tienen el mismo titulo que algun libro de nuestra base de datos pero distinto autor.
            final = 0.0
        else:
            final = 0.8 * titulo_score + 0.2 * colaborador_score


        filas_scored.append({
            "titulo_openalex": titulo_openalex,
            "colaboradores_openalex": colaboradores_openalex,
            "title_score": round(titulo_score, 4),
            "author_score": round(colaborador_score, 4),
            "final_score": round(final, 4),
            "id_openalex":id_openalex,
            "doi_openalex":doi_openalex
        })

    scored_df = pd.DataFrame(filas_scored).sort_values(
        ["final_score", "title_score", "author_score", "id_openalex", "doi_openalex"], ascending=False
    ).reset_index(drop=True)


    # ==============================
    # Mostrar TOP-5 y el mejor candidato
    # ==============================

    # print("\nTOP 5 (final_score, title_score, author_score, titulo_openalex)")
    #for i, row in scored_df.head(5).iterrows(): print(f"{i+1}. {row['final_score']} | {row['title_score']} | {row['author_score']} | {row['titulo_openalex']}")

    best = scored_df.iloc[0]
    second = scored_df.iloc[1] if len(scored_df) > 1 else None
    margin = round(best["final_score"] - (second["final_score"] if second is not None else 0.0), 4)

    # print("\nMEJOR CANDIDATO")
    # print("final_score:", best["final_score"], "title_score:", best["title_score"], "author_score:", best["author_score"])
    # print("titulo_openalex:", best["titulo_openalex"])
    # print("colaboradores_openalex:", best["colaboradores_openalex"])
    # print("margin_vs_2do:", margin)
    # print()


    # ==============================
    # Usamos el id_openalex del top 1 (solo si el final_score es mayor a 0.8) para extraer la cita de la API de OpenAlex
    # ==============================
    
    id_openalex_top_score = scored_df[scored_df["final_score"] >= 0.8]
    try:
        doi_openalex = scored_df["doi_openalex"].iloc[0]
        id_openalex_top_score = id_openalex_top_score["id_openalex"].iloc[0]
        id_openalex_top_score = re.findall(".org/(.+)", id_openalex_top_score)[0]
    except:
        cur.execute("UPDATE isbn SET openalex_consulted = ? WHERE titulo = ? AND colaboradores = ? ", ("except (score < 0.8)", titulo, colaboradores, ))
        con.commit()
        continue # Si no se encuentra un candidato que tenga la calidad suficiente (final_score mayor o igual a 0.8), pasamos a la siguiente iteracion (i.e., siguiente titulo de libro)

    # print(id_openalex_top_score)


    # Conexion con OpenAlex
    url = "https://api.openalex.org/" + id_openalex_top_score

    response = requests.get(url)
    data = response.json()

    ## print(json.dumps(data, indent=8, ensure_ascii=False))
    ## print()

    citacion = data["cited_by_count"]
    # print("cited_by_count: ", citacion)
    ## print(doi_openalex)
    print()
    print()
    print()
    print()
    print()


    # ==============================
    # Metemos la cita en la base de datos
    # ==============================

    cur.execute("UPDATE isbn SET openalex_consulted = ?, citacion = ?, doi_openalex = ? WHERE titulo = ? AND colaboradores = ? ", ("found", citacion, doi_openalex, titulo, colaboradores, ))
    con.commit()

    print()



In [ ]:
# scored_df.to_excel("scored_df.xlsx", index=False)
# scored_df

In [ ]:
##### Rellenando la columna "doi" de la tabla "editoriales" de la base de datos #####
# La logica a seguir es la siguiente: se rellena primero con los valores presentes en "doi_simeh", y luego, las celdas que queden vacías, se rellenan con los valores presentes en "doi_openalex"
for row in con.execute("SELECT id_isbn, doi_simeh, doi_openalex, doi FROM isbn"):
    id_isbn = row[0]
    doi_simeh = row[1]
    doi_openalex = row[2]
    doi = row[3]

    # print("id: ", id_isbn)
    # print("doi_simeh: ", doi_simeh)
    # print("doi_openalex: ", doi_openalex)
    # print()

    if doi_simeh is None or doi is not None:
        continue

    cur.execute("UPDATE isbn SET doi = ? WHERE id_isbn = ?", (doi_simeh, id_isbn, ))
    con.commit()

for row in con.execute("SELECT id_isbn, doi, doi_openalex FROM isbn"):
    id_isbn = row[0]
    doi= row[1]
    doi_openalex = row[2]

    if doi is None:
        cur.execute("UPDATE isbn SET doi = ? WHERE id_isbn = ?", (doi_openalex, id_isbn, ))
        con.commit()


In [ ]:
# Hacemos lo mismo pero para la columna "doi_scielo"
for row in con.execute("SELECT id_isbn, doi_scielo, doi_openalex, doi FROM isbn"):
    id_isbn = row[0]
    doi_scielo = row[1]
    doi_openalex = row[2]
    doi = row[3]

    # print("id: ", id_isbn)
    # print("doi_scielo: ", doi_scielo)
    # print("doi_openalex: ", doi_openalex)
    # print()

    if doi_scielo is None or doi is not None:
        continue

    cur.execute("UPDATE isbn SET doi = ? WHERE id_isbn = ?", (doi_scielo, id_isbn, ))
    con.commit()

for row in con.execute("SELECT id_isbn, doi, doi_openalex FROM isbn"):
    id_isbn = row[0]
    doi= row[1]
    doi_openalex = row[2]

    if doi is None:
        cur.execute("UPDATE isbn SET doi = ? WHERE id_isbn = ?", (doi_openalex, id_isbn, ))
        con.commit()

In [ ]:
##### Rellenando la columna "cantidad_isbn_doi" de la tabla "editoriales" de la base de datos #####
for row in cur.execute("SELECT nombre FROM editoriales").fetchall():
    editorial = row[0]

    cur.execute("""
        SELECT COALESCE(COUNT(NULLIF(TRIM(doi), '')), 0)
        FROM isbn
        WHERE (';' || REPLACE(REPLACE(editorial_principal_nombre, ' ;', ';'), '; ', ';') || ';')
              LIKE '%;' || TRIM(?) || ';%'
    """, (editorial,))
    
    # NOTAS:
        # COALESCE es una función SQL estándar que sirve para devolver el primer valor que no sea NULL de una lista de expresiones.
        # Ejemplo: "SELECT COALESCE(NULL, 5, 10)" -> en este caso devuelve 5, porque es el primer valor no nulo.
        # Por tanto, "SELECT COALESCE(SUM(...), 0)"" -> significa "si la suma (SUM(...)) da NULL (por ejemplo, porque no hay filas que cumplan el WHERE), devuelve 0 en su lugar.”

        # En SQL, el símbolo <> significa “distinto de” o “no igual a”.

    cantidad_isbn_doi = cur.fetchone()[0]

    # print(editorial, cantidad_isbn_doi)
    cur.execute("UPDATE editoriales SET cantidad_isbn_doi = ? WHERE nombre = ?", (cantidad_isbn_doi, editorial, ))
    con.commit()



In [ ]:
# Para exportar a excel una tabla de la base de datos sql
# pd.read_sql_query("SELECT * FROM isbn", con).to_excel("isbn.xlsx", index=False)

#### WorldCat API
https://developer.api.oclc.org/wcv2#/

Importante: parece que para acceder a esta API se necesita una suscripcion:

https://www.oclc.org/developer/api/oclc-apis/worldcat-search-api.en.html



In [ ]:
# Probando la WorldCat API
isbn = "[ISBN13_DEMO]"  # Sustituir por un ISBN público al ejecutar fuera del archivo histórico
limit = 50 # 50 es el maximo

url = "https://americas.discovery.api.oclc.org/worldcat/search/v2/bibs-holdings" # IMPORTANTE: CAMBIAR "americas" POR "europe/apac" para consultar otros continentes
#url = "https://americas.discovery.api.oclc.org/worldcat/search/v2/brief-bibs?q=isbn:[ISBN13_DEMO]"
params = {"isbn":isbn, "limit":limit}

response = requests.get(url, params=params)
print(response, response.reason)

In [ ]:
# Cerrar la conexion con la base de datos
#cur.close()
#con.close()

### 5) Limitaciones de la base de datos

Como todas las bases de datos que se alimentan distintas fuentes de información no estandarizadas, la calidad final de la base de datos, por mucho que se trate de limpiar, siempre vendrá limitada por los datos originales. A continuación, se exponen las limitaciones que se han identificado en la base de datos, para tenerlo en cuenta a la hora de realizar e interpretar los análisis y también con la esperanza de que algunas de estas limitaciones puedan resolverse en un futuro.


- Tabla **ventas**:
    - Estamos mezclando info de ventas de año 2024 (las que vienen del df “Librería Siglo”) con info de ventas de años 2020-2024 (las que vienen del df de “Scielo”)
- Tabla **isbn**
    - Columna **tipo_coedicion**:
        - Los datos que provienen de simeh y de la plantilla_propia si tienen estas variables (es decir, tendrán "si/no"), pero los datos provenientes de scielo no (es decir, son "None"). Esto quiere decir que en la base de datos habrá tres tipos de valores: si/no/None. Tenemos que tomar una decisión sobre como tratar los "None"
    - Columna **isbn13**:
        - A lo largo del proceso de análisis de los excel usados para alimentar esta base de datos (procedentes de las distintas instituciones), se han detectado inconsistencias en el isbn13: hay libros/documentos a los que se les ha asignado un codigo que no es isbn13 (pero se le ha adjudicado como isbn13). Esto no ocurre con frecuencia. Hemos eliminado estos libros/documentos de nuestra base de datos.
- Tabla **editoriales**
    - El nombre de una editorial puede aparecer escrito de distinta manera, lo que da lugar a duplicados.
        - ejemplos: Editorial Universitaria X - EUX / Editorial Universitaria X; Universidad Y (UY) / Universidad Y; Consejo Z / Consejo Z
- Tabla **autores**:
    - Columna **nombre**:
        - a veces aparecen valores como "varios autores"
        - autores que a veces tienen asociado texto entre paréntesis. Esto da lugar a que un mismo autor sea considerado como dos autores distintos.
            - ejemplo: "[NOMBRE COMPLETO] (Compilador) / [NOMBRE COMPLETO]"
        - autores que a veces aparecen con un apellido y otras con los dos. Da lugar al mismo problema mencionado arriba.
            - ejemplos: "[NOMBRE ABREVIADO] / [NOMBRE COMPLETO]"; "[NOMBRE CON UN APELLIDO] / [NOMBRE CON DOS APELLIDOS]"; "[NOMBRE ABREVIADO] / [NOMBRE COMPLETO]"
        - autores que a veces aparecen con nombramientos como "Ph. D", "Msc", "Arq" y otras veces no. Da lugar al mismo problema mencionado arriba.
            - ejemplos:  "[NOMBRE] Ph. D / [NOMBRE]"; "Arq. [NOMBRE] / [NOMBRE]"


### 6) Consultando a la base de datos. Ejemplos

In [ ]:
# Prueba consulta SQL
# Vamos a hacer una tabla que nos muestre los 5 autores que tienen mayor cantidad de isbn13 asociados (es decir, una tabla de dos columnas: Nombre autor, Cantidad_isbn13)
query = """
SELECT 
    autores.nombre AS Autor, 
    COUNT(DISTINCT isbn_autores.id_isbn) AS Cantidad_isbn13
FROM isbn_autores
JOIN autores 
    ON autores.id_autor = isbn_autores.id_autor
GROUP BY isbn_autores.id_autor, autores.nombre
ORDER BY Cantidad_isbn13 DESC, Autor ASC
LIMIT 5;
"""

for row in cur.execute(query):
    print(row)




In [ ]:
# Bonito (usando tabulate)
data = [row for row in cur.execute(query)]
headers = ["Autor", "Cantidad_isbn13"]
print(tabulate.tabulate(data, headers,"grid"))


In [ ]:
# Prueba consulta SQL
# Vamos a hacer una tabla que nos muestre los 5 autores que tienen mayor cantidad de isbn13 asociados, y los titulos de libros asociados (es decir, una tabla de tres columnas: Nombre autor, Cantidad_isbn13, Titulo_libro)
# NOTA!!!! REALMENTE CUENTA LOS ISBN DISTINTOS, NO LOS LIBROS (TITULOS) DISTINTOS

query = """
SELECT
  autores.nombre AS Autor,
  COUNT(DISTINCT isbn_autores.id_isbn) AS Cantidad_isbn13,
  REPLACE(GROUP_CONCAT(DISTINCT isbn.titulo), ',', '; ') AS Titulo_libro
FROM isbn_autores
JOIN autores ON autores.id_autor = isbn_autores.id_autor
JOIN isbn ON isbn.id_isbn = isbn_autores.id_isbn
GROUP BY isbn_autores.id_autor, autores.nombre
ORDER BY Cantidad_isbn13 DESC, Autor ASC
LIMIT 5;
"""

for row in cur.execute(query):
    print(row)

In [ ]:
# Bonito (usando tabulate)
data = [row for row in cur.execute(query)]
headers = ["Autor", "Cantidad_isbn13", "Titulo_libro"]
print(tabulate.tabulate(data, headers,"grid"))


In [ ]:
# Prueba consulta SQL
# Vamos a hacer una tabla que nos muestre los 5 autores que tienen mayor cantidad de libros asociados, y los titulos del libro (es decir, una tabla de tres columnas: Nombre autor, Num_libros, Titulo_libros)

query = """
SELECT
  autores.nombre AS Autor,
  COUNT(DISTINCT libros_autores.id_libro) AS Num_libros,
  REPLACE(GROUP_CONCAT(DISTINCT libros.titulo), ',', '; ') AS Titulo_libro
FROM libros_autores
JOIN autores ON autores.id_autor = libros_autores.id_autor
JOIN libros ON libros.id_libro = libros_autores.id_libro
GROUP BY libros_autores.id_autor, autores.nombre
ORDER BY Num_libros DESC, Autor ASC
LIMIT 5;
"""

for row in cur.execute(query):
    print(row)


In [ ]:
# Bonito (usando tabulate)
data = [row for row in cur.execute(query)]
headers = ["Autor", "Num_libros", "Titulo_libro"]
print(tabulate.tabulate(data, headers,"grid"))

In [ ]:
# Prueba consulta SQL
# Vamos a hacer una tabla que nos muestre el id_libro, el titulo del libro, los isbn13 asociados, y los autores que han colaborado en ese libro

query = """
WITH isbns AS (
  SELECT
    il.id_libro,
    GROUP_CONCAT(i.isbn13, '; ') AS isbn13_asociados
  FROM isbn_libros AS il
  JOIN isbn AS i ON i.id_isbn = il.id_isbn
  WHERE COALESCE(TRIM(i.isbn13), '') <> ''
  GROUP BY il.id_libro
),
aut AS (
  SELECT
    la.id_libro,
    GROUP_CONCAT(a.nombre, '; ') AS autores
  FROM libros_autores AS la
  JOIN autores AS a ON a.id_autor = la.id_autor
  WHERE COALESCE(TRIM(a.nombre), '') <> ''
  GROUP BY la.id_libro
)
SELECT
  l.id_libro,
  l.titulo AS titulo_libro,
  isbns.isbn13_asociados,
  aut.autores
FROM libros AS l
LEFT JOIN isbns ON isbns.id_libro = l.id_libro
LEFT JOIN aut   ON aut.id_libro   = l.id_libro
WHERE COALESCE(TRIM(l.titulo), '') <> ''
ORDER BY l.id_libro;
"""

for row in cur.execute(query):
    print(row)

In [ ]:
# Bonito (usando tabulate)
data = [row for row in cur.execute(query)]
headers = ["id_libro", "titulo_libro", "isbn13_asociados", "autores"]
print(tabulate.tabulate(data, headers,"grid"))